# C-test ONE · Results

*Ctest-Results-v1.57.0*

Produces the tables and figures of the paper for one reference machine: accuracy over
the (retrieval, transformation) grid, correlations between the measures and accuracy, agent
characteristic curves and abilities, and the infra- and supra-intelligence windows.

| reads | writes |
|---|---|
| `ctest_battery.json`, `ctest_metrics.json`, `ctest_results.json`, `ctest_calibration.json`, `ctest_pricing_<machine>.json` | figures (PNG, PDF) and LaTeX tables |

**Needs** nothing but the data. CPU only. The machine is `MACHINE_ID` and `MACHINE_TAG`.

## Running it

**Locally.** From the repository root, install the requirements and open the notebook in Jupyter:

```
pip install -r requirements.txt
jupyter lab notebooks/
```

API keys are read from environment variables (see `.env.example`). On the first local run the
notebook copies `data/` into a `work/` folder and runs there, so the released data are never
modified; set `CTEST_WORK_DIR` to use another folder.

**Colab.** Upload the data files to `MyDrive/ctest` and mount Drive; keys go in the Secrets panel.

**Kaggle.** Attach a dataset holding the data files and set `KAGGLE_DATASET` to its name; keys go in
Add-ons, Secrets. With Kaggle credentials the notebook can also push its results back as a new
version of the dataset.


In [ ]:
# Local runs work in a copy of the data, so that data/ stays exactly as released.
# On Colab and Kaggle this cell does nothing.
import os, glob, shutil
if not (os.path.exists("/kaggle") or os.path.exists("/content")):
    _root = os.getcwd()
    if os.path.basename(_root) == "notebooks":
        _root = os.path.dirname(_root)
    try:
        from dotenv import load_dotenv          # optional: keys from a .env file
        load_dotenv(os.path.join(_root, ".env"))
    except ImportError:
        pass
    WORK_DIR = os.path.abspath(os.environ.get("CTEST_WORK_DIR", os.path.join(_root, "work")))
    os.makedirs(WORK_DIR, exist_ok=True)
    for _f in glob.glob(os.path.join(_root, "data", "*")):
        _dst = os.path.join(WORK_DIR, os.path.basename(_f))
        if not os.path.exists(_dst):
            shutil.copy(_f, _dst)
    os.chdir(WORK_DIR)
    print(f"local run: working in {WORK_DIR}")


## Which items these numbers are about

Three filters, in order, and only the first is independent of the reference
machine.

Validity belongs to the rule and the
executor, so these 45 stay valid whichever machine prices them.

**30 certified under Qwen3-8B.** Unquestionable at $\delta$: no rival inside
the budget reproduces the prefix and then diverges.
This is machine-relative
and a different $\M$ gives a different 30.

**16 compressible.** A program the machine could execute cost less than writing
the continuation out, so $h$ is the cost of a rule rather than of a copy.
Also
machine-relative.

The tables below are over the 30 and the 16.
Before the validity filter existed
the same battery gave 54 and 21, and those numbers appear in earlier drafts.

## 1 · Load

In [ ]:
import os, re, json, glob, math, collections, statistics as st
import matplotlib
import matplotlib.pyplot as plt
import numpy as np

BATTERY = "ctest_battery.json"
METRICS = "ctest_metrics.json"
RESULTS = "ctest_results.json"
CALIB   = "ctest_calibration.json"
MACHINE_ID  = "qwen3-14b-4bit-think"    # names h in the metrics file; qwen3-8b-4bit-think or phi-4-reasoning
MACHINE_TAG = "qwen3-14b"               # names n_M and K in the battery; qwen3-8b or phi-4-reasoning

PRICING = f"ctest_pricing_{MACHINE_TAG}.json"
FIGDIR  = "."                           # where the pngs and tex go

def _find(f):
    for p in [f, f"/content/{f}"] + glob.glob(f"/kaggle/input/**/{f}",
                                              recursive=True):
        if os.path.exists(p): return p
    return None

def _load(f, what):
    p = _find(f)
    if not p:
        print(f"  {f:32s} NOT FOUND  ({what})")
        return None
    print(f"  {f:32s} {p}")
    return json.load(open(p))

print("files")
_bat = _load(BATTERY, "the battery; nothing works without it")
_pri = _load(PRICING, "rings fall back to the bare")
_met = _load(METRICS, "the h figures and table are skipped")
_res = _load(RESULTS, "only the battery figure is drawn")
_cal = _load(CALIB, "the necessary-condition line falls back to defaults")

BAT = _bat["items"] if _bat else []
PRI = (_pri or {}).get("items", _pri or {})
MET = (_met or {}).get("items", _met or {})
ROWS = (_res or {}).get("rows", [])

# An examinee whose calls time out has a timeout rate, not a score, and
# reporting it beside real ones invites a comparison that is not there. Any
# examinee whose replies are mostly missing is dropped from the figures and
# tables, and named.
ROWS_TWINS = [r for r in ROWS if r["examinee"] not in {"Qwen/Qwen3-14B", "Qwen/Qwen3-8B", "microsoft/Phi-4-reasoning"}]   # before the twin drop, for the refusals table

# A MODEL WITH A `-fallback` TWIN IS DROPPED HERE, once, for everything. The
# two are near-copies -- the fallback run differs only on the items where the
# model declined -- so keeping both put that model into every mean twice: the
# `all` column of a correlation table, the height of a point in a shift
# figure, the ensemble a window is fitted on. It was being filtered in six
# places and missed in a seventh, which is what a filter at the point of use
# invites.
#
# The refusals table is the exception and reads ROWS_ALL_RAW, because a
# refusal is the thing it reports and the pair is the point there.
_EX = {r["examinee"] for r in ROWS}
_TWIN = {e[:-len("-fallback")] for e in _EX if e.endswith("-fallback")}

# A REFERENCE MACHINE SITTING THE BATTERY IS NOT AN EXAMINEE HERE. Its rows
# come from the self-measurement experiment, which is reported on its own
# in an appendix: the machine chose which items certify, so its score on
# them is not comparable with the others'. Dropped for everything, the
# refusals table included, since it is not part of that comparison either.
REFERENCE_EXAMINEES = {"Qwen/Qwen3-14B", "Qwen/Qwen3-8B",
                       "microsoft/Phi-4-reasoning"}
_REF = _EX & REFERENCE_EXAMINEES
if _REF:
    ROWS = [r for r in ROWS if r["examinee"] not in _REF]
    print(f"  dropped {len(_REF)} reference machine(s) sitting as examinees, "
          f"reported separately: " + ", ".join(sorted(_REF)))
    _EX = _EX - _REF
if _TWIN:
    ROWS = [r for r in ROWS if r["examinee"] not in _TWIN]
    print(f"  dropped {len(_TWIN)} model(s) whose -fallback twin is present: "
          + ", ".join(sorted(_TWIN)))

ROWS_ALL_RAW = list(ROWS_TWINS)   # reset below if any examinee is dropped
DROP_IF_EMPTY = 0.5
_bad = {}
for _e in sorted({r["examinee"] for r in ROWS}):
    _rr = [r for r in ROWS if r["examinee"] == _e]
    _em = sum(1 for r in _rr if not r.get("answer"))
    if _em > DROP_IF_EMPTY * len(_rr):
        _to = sum(1 for r in _rr if "Timeout" in (r.get("raw") or ""))
        _bad[_e] = (_em, len(_rr), _to)
if _bad:
    print()
    for _e, (_em, _n, _to) in _bad.items():
        print(f"  DROPPED {_e}: {_em} of {_n} replies empty"
              + (f", {_to} of them timeouts" if _to else "")
              + ". That is a timeout rate, not a score.")
    ROWS_ALL_RAW = [r for r in ROWS_TWINS if r["examinee"] not in _bad]
    ROWS = [r for r in ROWS if r["examinee"] not in _bad]

CAL  = _cal or {}

VALID_ONLY = True

# A "-fallback" examinee is the same model asked for server-side fallbacks
# where a refusal is recommended. A fallback answer is not the model's own
# answer, so it is a different configuration and does not belong in a table
# that pools examinees: it would sit beside its own plain version under a name
# a reader cannot tell apart, and the ensemble would average a model with
# itself.
#
# It also breaks the item count. A correlation table keeps the items EVERY
# examinee answered, and the fallback run covered a different set, so three
# items fell out of the certified table and it read 13 instead of 16.
#
# They are kept in the file and reported on their own.
DROP_FALLBACK_EXAMINEES = False
                          # a "-fallback" examinee is the same model asked for
                          # server-side fallbacks where a refusal is
                          # recommended. It is a different configuration, so it
                          # gets its own row rather than being pooled with the
                          # plain one, and the short label ends in -fb.

SCORE_KIND = "full"
FIG_SUFFIX = ("" if SCORE_KIND == "prefix" else "_" + SCORE_KIND) \
             + "_" + MACHINE_TAG
SCORE_SHOWN = {"prefix": "prefix%", "full": "full"}[SCORE_KIND]

# HOW THE MACHINE IS WRITTEN IN A ROW LABEL. Plain, because every table here
# is for one machine and its caption says which: the family subscript repeats
# that on every row and makes each label longer than the quantity it names.
# The family form is kept for prose comparing the two and for the shift
# figures, where both are on the page, family first so that 8 and 14 go on
# meaning something once a machine from another family appears.
MACHINE_SUB = r"\mathcal{M}"
MACHINE_FAMILY = {"phi-4-reasoning": r"\mathcal{M}_{\mathrm{p4}}",
                  "qwen3-8b": r"\mathcal{M}_{\mathrm{q8}}",
                  "qwen3-14b": r"\mathcal{M}_{\mathrm{q14}}"}.get(
                      MACHINE_TAG, r"\mathcal{M}")

def ACC(row):
    """What one attempt scored, under the marking this run is using."""
    f = row["n_correct"] / row["k"]
    return f if SCORE_KIND == "prefix" else (1.0 if f >= 0.999 else 0.0)

# How the heatmap panels are wrapped. Eleven in a row is two metres of figure
# at a readable size, so nothing is readable at the size a page allows.
HEAT_ROWS = [6, 6]        # panels per row of the heat figure

# The reference machine, for captions and titles. Certification and
# compressibility are relative to it, so every table and figure that rests on
# either has to say which machine. MACHINE_ID is set further up; this is the
# short form a caption uses.
MACHINE_SHOWN = {"phi-4-reasoning": "Phi-4-reasoning",
                 "qwen3-8b-4bit-think": "Qwen3-8B",
                 "qwen3-14b-4bit-think": "Qwen3-14B",
                 "olmo3-7b-4bit-think": "OLMo-3-7B"}.get(MACHINE_ID, MACHINE_ID)

def _is_valid(b):
    """The seed, run cold a second time, reproduced what the first run made.

    RECORDED TWO WAYS, and both mean this. Generation writes `valid` and
    `valid_votes` as it admits an item; the standalone extension pass writes
    `repro.seed_cold.verdict`. Reading only the second dropped every item made
    since the check moved into generation, which was nine of them and all of
    the recent ones: the battery reported 47 valid where it has 54.
    """
    return (b.get("valid") is True
            or (b.get("repro", {}).get("seed_cold") or {}).get("verdict")
            == "reproduced")

_valid = {b["id"] for b in BAT if _is_valid(b)}
print(f"{len(BAT)} generated, {len(_valid)} valid"
      + ("   using the valid only" if VALID_ONLY
         else "   VALID_ONLY is off, using all of them"))

def _cert_of(S, pid=None):
    """The certification record of the program that WON, or of `pid`.

    Two faults it exists to prevent. The winner is named by `p_star`, and
    scanning `programs` for the first entry with a passing `cert` returns
    whichever the dict happens to yield first, which is not the cheapest and
    not what K_cond was taken from. And a re-annotation moves the previous
    `cert` to `cert_prev` and leaves `cert` absent until the item is run
    again, so a scan finds nothing and the item reads as uncertified when it
    is not: v17-0004 was drawn as a crossed point at the 8B value because of
    this, when it certifies under the 14B at 225 bits.
    """
    P = S.get("programs") or {}
    r = P.get(pid or S.get("p_star")) or {}
    return r.get("cert") or r.get("cert_prev") or {}

def _tau_of(S):
    """The tokens the reference machine spent on the program that certified."""
    return _cert_of(S).get("tau")

ITEMS = {}
ITEMS_ALL = {}          # every item, so the invalid ones can still be listed
for b in BAT:
    me = (MET.get(b["id"]) or {}).get(MACHINE_ID) or {}
    pr = PRI.get(b["id"]) or {}
    kc, ku = me.get("K_cond"), me.get("K_uncond")
    ps = str(pr.get("p_star") or "")
    # r and t are read from p_star, the description that achieved K, and each
    # machine picks its own, so they belong to a machine exactly as K and n_M
    # do. The plain fields hold whichever run measured last, which is not this
    # machine unless it happens to have been second.
    _rt = {k: b.get(f"{k}_{MACHINE_TAG}", b.get(k))
           for k in ("r_hat", "t_hat", "r_hat_sd", "t_hat_sd",
                     "r_hat_all", "t_hat_all", "typed_from")}
    _rec = dict(
        b, **_rt,
        K=b.get(f"K_{MACHINE_TAG}"), n_M=b.get(f"n_M_{MACHINE_TAG}"),
        p_star_len=len(ps) or None, p_star=pr.get("p_star"),
        h=me.get("h"), K_cond=kc, K_uncond=ku, c_xpref=me.get("c_xpref"),
        band=me.get("band"),
        prefix_pays=(ku - kc) if kc is not None and ku is not None else None,
        # conditional Levin complexity: what the description costs to state
        # plus the log of what it costs to run. Put on the item so it can be
        # binned and plotted like h and K, rather than recomputed at each use.
        Kt=(kc + math.log2(_tau_of(me))) if (kc is not None and _tau_of(me))
           else None,
        load=None if b.get("r_hat") is None else b["r_hat"] + b["t_hat"],
        # tokens the REFERENCE machine spent on the program that certified the
        # item: work rather than code length, and the only compute cost among
        # the measures
        tau=_tau_of(me))
    ITEMS_ALL[b["id"]] = _rec
    if VALID_ONLY and b["id"] not in _valid: continue
    ITEMS[b["id"]] = _rec

if DROP_FALLBACK_EXAMINEES:
    _nf = len(ROWS)
    _fb_names = sorted({r["examinee"] for r in ROWS
                        if r["examinee"].endswith("-fallback")})
    ROWS = [r for r in ROWS if not r["examinee"].endswith("-fallback")]
    ROWS_ALL_RAW = [r for r in ROWS_ALL_RAW
                    if not r["examinee"].endswith("-fallback")]
    if _fb_names:
        print(f"  {_nf - len(ROWS)} row(s) set aside from "
              f"{', '.join(_fb_names)}: a fallback answer is not the model's "
              f"own answer, so it is reported separately")

# The results have to be filtered the same way, or the score tables count
# attempts on items the analysis has excluded and report "of 54" under a
# heading that says 30.
if VALID_ONLY:
    _n0 = len(ROWS)
    ROWS = [r for r in ROWS if r["item"] in ITEMS]
    print(f"  {len(ROWS)} of {_n0} result rows are on valid items")

# CERTIFIED UNDER THIS MACHINE. The results cover every item that certified
# under ANY machine, because the evaluator asks for those: unquestionability is
# machine-relative and an item one machine certifies and the other does not
# still has to be answered for the analysis under the first. But a table headed
# "unquestionable under Qwen3-14B" must count only those, or the caption says
# 27 while the "ever right" column says "of 30".
_n2 = len(ROWS)
ROWS = [r for r in ROWS if ITEMS[r["item"]].get("n_M") is not None]
if len(ROWS) < _n2:
    print(f"  {_n2 - len(ROWS)} row(s) dropped: on items {MACHINE_SHOWN} did "
          f"not certify, so they are not part of this machine's battery")

_n1 = len(ROWS)
_need = {i: (ITEMS[i].get("n_M") or 0) for i in ITEMS}
_best = {}
for r in ROWS:
    _n = r.get("n_M")
    if _n is None or _n < _need.get(r["item"], 0): continue
    _k = (r["examinee"], r["item"])
    _b = _best.get(_k)
    if _b is None or _n < _b: _best[_k] = _n
ROWS = [r for r in ROWS
        if r.get("n_M") == _best.get((r["examinee"], r["item"]))]
if len(ROWS) < _n1:
    print(f"  {_n1 - len(ROWS)} row(s) dropped: answered at a prefix shorter "
          f"than {MACHINE_SHOWN} needs, or at a longer one when a usable "
          f"shorter reply exists")

_cert = [i for i, v in ITEMS.items() if v.get("n_M") is not None]
_hh   = [i for i, v in ITEMS.items() if v.get("h") is not None]
_ex   = sorted({r["examinee"] for r in ROWS})
print(f"\n{len(ITEMS)} items, {len(_cert)} certified, {len(_hh)} with an h")
print(f"{len(ROWS)} result rows from {len(_ex)} examinee(s): {', '.join(_ex) or 'none'}")


## 2 · The battery

Every item where it was MEASURED, not where it was asked for, so drift shows as distance from a grid line. The filled circle is $K$; the ring is the length of the same description, so the two can be compared. Circles shrink as the battery grows, or the picture silts up.

In [ ]:
def plot_battery(items, pricing=None, path=None, bits_per_char=None,
                 show_ids=False):
    """Every item at its TYPED position, with two circles: K and the length of
    its rule in bits.

    x is t_hat, y is r_hat, so an item sits where it was MEASURED rather than
    where it was asked for. "Retrievals" throughout, not "tables": a text, an
    ordering and a lookup table are all things that must be remembered, and
    calling them tables reads as though only one shape counted. The asked cells are the grid lines behind, which
    makes the drift visible as distance from the nearest intersection.

    The two circles are the point:

      outer, filled     K, what the reference machine charges for the cheapest
                        program that generates the sequence
      inner, hollow     the length in bits of THE SAME DESCRIPTION, at
                        bits_per_char. That is p_star, the description that
                        achieved K, taken from the pricing file.

                        It has to be the same description or the two circles
                        measure different things. Drawing K against the BARE's
                        length does exactly that on two thirds of the battery,
                        since p_star is the bare on only 21 of 63 items, and
                        the ratio then runs from 0.70 to 22.55 instead of 0.46
                        to 1.85. Most of that spread was the mismatch, not a
                        property of any item.

    "Rule" is avoided here because the notebooks use three words for three
    different things: the SEED is what the composer first wrote, the BARE is
    the shortest verified equivalent, and the GLOSS is prose describing the
    item. Only the bare is priced.

    An item with NO ring has no bare: nothing was ever verified to reproduce
    its sequence, so there is no length to draw. It can still have a fill,
    because K came from the pricing pass, which certifies against the
    structural rival whether or not a composer program was verified. Eight of
    eighty are like this, and they are worth spotting.

    When they coincide, the proxy predicted the price. When the outer is much
    larger, the machine charges more than the characters suggest, which is
    where a short rule is expensive to state in this machine's terms. An item
    with no bare has no inner circle at all: nothing was verified to price.
    """
    import matplotlib.pyplot as plt
    from matplotlib.lines import Line2D
    import numpy as np

    have = [b for b in items if b.get("r_hat") is not None]
    if not have:
        print("no r_hat on any item; run the retype pass first"); return

    pricing = pricing or {}

    def _described(b):
        """The string K refers to, and its length. p_star when there is one."""
        p = (pricing.get(b["id"]) or {}).get("p_star")
        if p: return len(p), "p_star"
        if b.get("bare"): return len(b["bare"]), "bare"
        return 0, "none"

    if bits_per_char is None:
        rr = sorted(b[f"K_{MACHINE_TAG}"] / _described(b)[0] for b in items
                    if b.get(f"K_{MACHINE_TAG}") and _described(b)[0])
        bits_per_char = rr[len(rr)//2] if rr else 1.80
    # `or 0` would turn an unpriced item into a zero-cost one, and rad(0) is
    # still a visible dot, so the reader cannot tell "costs almost nothing"
    # from "no $K$ measured". Kept apart and drawn differently.
    kmax = max([b[f"K_{MACHINE_TAG}"] for b in have
                if b.get(f"K_{MACHINE_TAG}")] or [1])

    xs = [b["t_hat"] for b in have]; ys = [b["r_hat"] for b in have]
    # room on the right for the legend, which sits there rather than top-left
    # because the items crowd the left of the grid: r=0 and t=0 are cells and
    # a legend over them hides data
    x0, x1 = -0.7, max(6.5, max(xs) + 0.9) + 1.9
    y0, y1 = -0.7, max(2.9, max(ys) + 0.7)
    # equal aspect, or the circles are ellipses and the two radii cannot be
    # compared by eye, which is the whole point of drawing two
    fig, ax = plt.subplots(figsize=(12, 12 * (y1 - y0) / (x1 - x0)))
    ax.set_aspect("equal")
    for x in range(7): ax.axvline(x, color="0.92", lw=1, zorder=0)
    for y in range(3): ax.axhline(y, color="0.92", lw=1, zorder=0)

    _shrink = (80.0 / max(len(have), 20)) ** 0.5
    def rad(bits):
        return (0.05 + 0.30 * (max(bits, 0) / kmax) ** 0.5) * _shrink

    # items pile up on the same typed position, so nudge them apart on a small
    # deterministic spiral. The nudge is under a fifth of a unit and the grid
    # lines are still where the asked cells are.
    import collections as _c, math as _m
    at = _c.defaultdict(list)
    for b in have: at[(round(b["t_hat"], 2), round(b["r_hat"], 2))].append(b)
    pos = {}
    for (x, y), bs in at.items():
        if len(bs) == 1:
            pos[bs[0]["id"]] = (x, y); continue
        for k, b in enumerate(bs):
            a = 2 * _m.pi * k / len(bs)
            r = (0.10 + 0.05 * (k // 8)) * _shrink
            pos[b["id"]] = (x + r * _m.cos(a), y + r * _m.sin(a))

    for b in have:
        x, y = pos[b["id"]]
        # THIS MACHINE, not a hard-coded one. Both fields were read from
        # _qwen3-8b whatever the figure was named for, so the 14B figure was
        # drawn from 8B numbers and eight items appeared to have no K when
        # under the 14B every valid item has one.
        k = b.get(f"K_{MACHINE_TAG}")
        priced = k is not None
        unq = b.get(f"n_M_{MACHINE_TAG}") is not None
        cert = unq and (MET.get(b["id"], {}).get(MACHINE_ID, {})
                        .get("band") not in (None, "p_lam"))
        col = ("#1f77b4" if cert            # blue,   certified
               else "#7fb3d5" if unq        # pale,   unquestionable, not
               else "#d62728" if priced     # red,    questionable
               else "#111111")              # black,  no K measured
        if priced:
            ax.add_patch(plt.Circle((x, y), rad(k), color=col, alpha=0.16,
                                    lw=0, zorder=2))
        else:
            # no K, so no filled circle at all. A cross marks the position so
            # the item is not simply absent from the picture.
            ax.plot([x], [y], "x", color=col, ms=7 * _shrink ** 0.5, mew=1.4,
                zorder=4)
        ch, _src = _described(b)
        if ch:
            ax.add_patch(plt.Circle((x, y), rad(ch * bits_per_char),
                                    fill=False, ec=col, lw=1.1, zorder=3))
        if priced:
            ax.plot([x], [y], ".", color=col, ms=3, zorder=4)
        if show_ids:
            ax.annotate(b["id"].split("-")[1].lstrip("0"), (x, y),
                        fontsize=5, color="0.3", ha="center", va="center",
                        zorder=5)

    # mathtext, so the hat sits on the letter as it does in the write-up
    ax.set_xlabel(r"$\hat{t}$,  transformations measured", fontsize=11)
    ax.set_ylabel(r"$\hat{r}$,  retrievals measured", fontsize=11)
    # one decimal on both axes, since the measurements are means of several
    # typings and land between the integers the design uses
    from matplotlib.ticker import FormatStrFormatter, MultipleLocator
    ax.xaxis.set_major_locator(MultipleLocator(1))
    ax.yaxis.set_major_locator(MultipleLocator(1))
    ax.xaxis.set_major_formatter(FormatStrFormatter("%.1f"))
    ax.yaxis.set_major_formatter(FormatStrFormatter("%.1f"))
    ax.set_xlim(x0, x1); ax.set_ylim(y0, y1)
    _nof = sum(1 for b in have if b.get(f"K_{MACHINE_TAG}") is None)
    ax.set_title(
        f"{len(have)} valid items at their measured position, "
        f"under {MACHINE_SHOWN}\n"
        f"disc: $K$    ring: the same description at "
        f"{bits_per_char:.2f} bits a character"
        + (f"    ({_nof} have no $K$, drawn as a cross)" if _nof else ""),
        fontsize=10, loc="left")
    # THE THREE FILTERS BY NAME, and nothing else called anything else. The
    # words valid, unquestionable and certified mean one thing each here as
    # they do everywhere: every item drawn is valid, the colour says how far
    # past that it got, and the two circles are a size comparison and not a
    # fourth filter.
    ax.legend(handles=[
        Line2D([], [], marker="o", ls="", mfc="#1f77b4", mec="none",
               alpha=.45, ms=12, label="certified under this machine"),
        Line2D([], [], marker="o", ls="", mfc="#7fb3d5", mec="none",
               alpha=.45, ms=12, label="unquestionable, not certified"),
        Line2D([], [], marker="o", ls="", mfc="#d62728", mec="none",
               alpha=.45, ms=12, label="questionable"),
        Line2D([], [], marker="x", ls="", mec="#111111", mew=1.6, ms=8,
               label="no $K$ measured, so no disc"),
        Line2D([], [], ls="", label=""),
        Line2D([], [], marker="o", ls="", mfc="none", mec="0.3",
               ms=11, label="ring: the description written out"),
        Line2D([], [], ls="", label="ring wider than the disc: the machine"),
        Line2D([], [], ls="", label="   finds the description unsurprising"),
        Line2D([], [], ls="", label="disc wider than the ring: it does not")],
        loc="upper left", bbox_to_anchor=(1.01, 1.0), fontsize=8,
        frameon=False, borderaxespad=0)
    fig.tight_layout(rect=(0, 0, 0.78, 1))
    if path:
        fig.savefig(path, dpi=150); print(f"wrote {path}")
    return fig


In [ ]:
# the valid items, not the raw battery: BAT is every item the composer ever
# produced, and the grid should show what the analysis is about.
_BAT_V = [b for b in BAT if b["id"] in ITEMS]
print(f"the grid shows the {len(_BAT_V)} valid items, not all {len(BAT)}")
fig = plot_battery(_BAT_V, pricing=PRI)
plot_battery(_BAT_V, pricing=PRI, path=f"{FIGDIR}/fig_battery_{MACHINE_TAG}.png")
plot_battery(_BAT_V, pricing=PRI, path=f"{FIGDIR}/fig_battery_ids_{MACHINE_TAG}.png",
             show_ids=True)


## 3 · Accuracy over the grid

One panel per examinee. The count is printed under each value for that reason.

In [ ]:
ENSEMBLE = "__all__"   # the column name for the mean over examinees;
                       # defined here because short() reads it and the
                       # figures below use short().
# short() is used by the figures below as well as by the tables, so it
# is defined here rather than beside the first table that needs it.
def short(name):
    """A short column label. The caption says what each one is.

      gemini-3.5-flash-lite        ->  g3.5FL
      claude-haiku-4-5-20251001    ->  haiku
      claude-sonnet-5              ->  sonnet
      gpt-5-nano                   ->  gpt5na
      allenai/OLMo-2-1124-7B       ->  olmo2
      Qwen/Qwen2.5-1.5B-Instruct   ->  q1.5B
      claude-opus-5-fallback       ->  opus*

    Short enough that a table of nine examinees and a `needs' column does not
    run off the page, which is what a longer scheme did.
    """
    if name == ENSEMBLE:
        return "all"
    # "-fallback" is the same model with server-side fallbacks asked for. It
    # has to be told apart from the plain one or a table shows two rows called
    # "opus" and a reader cannot say which is which.
    _fb = name.endswith("-fallback")
    if _fb:
        return short(name[:-len("-fallback")]) + "*"
    n = name.split("/")[-1]
    m = re.match(r"gemini-([0-9.]+)-(.+)", n)
    if m:
        return "g" + m.group(1) + "".join(w[0].upper()
                                          for w in m.group(2).split("-"))
    m = re.match(r"claude-([a-z]+)", n, re.I)
    if m:
        return m.group(1).lower()
    m = re.match(r"gpt-([0-9]+)[.0-9]*-([a-z]+)", n, re.I)
    if m:
        # `sol' is a name and survives whole; everything else is cut to two,
        # which is enough to tell nano from mini.
        _suf = m.group(2).lower()
        return f"gpt{m.group(1)}{_suf if _suf == 'sol' else _suf[:2]}"
    m = re.match(r"OLMo-([0-9]+)", n, re.I)
    if m:
        return f"olmo{m.group(1)}"
    m = re.match(r"Qwen[0-9.]*-([0-9.]+B)", n, re.I)
    if m:
        return f"q{m.group(1)}"
    m = re.match(r"gemma-([0-9]+)", n, re.I)
    if m:
        return f"gemma{m.group(1)}"
    return n[:12]

# short() and ENSEMBLE are defined here as well as in the tables cell,
# so the figures do not depend on the order the cells are run in.

import re
import math

def by_capacity(names, results=None):
    """Examinees weakest first.

    By MEASURED accuracy when the results are to hand, which works for any
    family and is what a reader wants: the columns then read left to right as
    the ladder. Sorting on the version number only worked inside one family,
    and put two open-weight models, the weakest in the set, at the end.

    Falls back to version and tier when nothing has been measured yet.
    """
    names = list(names)
    if results:
        sc = collections.defaultdict(lambda: [0, 0])
        for r in results:
            s = sc[r["examinee"]]
            s[0] += r["n_correct"]; s[1] += r["k"]
        if all(sc.get(n, [0, 0])[1] for n in names):
            return sorted(names, key=lambda n: sc[n][0] / max(sc[n][1], 1))

    def key(n):
        s = n.split("/")[-1]
        m = re.match(r"gemini-([0-9.]+)-(.*)", s)
        if not m:
            return (1, 0.0, 0, s)
        return (0, float(m.group(1)), 0 if "lite" in m.group(2) else 1, s)
    return sorted(names, key=key)

def certified_checks(items, met, machine_id, tag, pri,
                     shown=None, path=None):
    """Two things that hold of every certified item, neither of them imposed.

    The winner sits inside the unquestionability limit, and conditioning makes
    the description below the cost of describing the whole sequence. The second is
    sufficient for the first and not necessary, so an item can fail it and
    still be certified; reporting both says which.
    """
    n = ok1 = ok2 = 0
    f1, f2 = [], []
    for i, b in items.items():
        if not _is_valid(b):
            continue
        S = (met.get(i) or {}).get(machine_id) or {}
        if S.get("band") in (None, "p_lam"):
            continue
        kc = S.get("K_cond")
        pr = pri.get(i) or {}
        if kc is None or not pr.get("beta"):
            continue
        L = b.get(f"n_M_{tag}")
        n += 1
        lim = pr["beta"] * L + pr["c0"] - pr["delta"]
        if kc < lim: ok1 += 1
        else: f1.append((i, kc, lim))
        if kc < pr["K"]: ok2 += 1
        else: f2.append((i, kc, pr["K"]))
    print(f"\ncertified under {shown or tag}: {n} of these have a beta, "
          f"so the checks are over {n} items")
    print(f"  K_cond < beta*L_u + c0 - delta : {ok1}/{n}"
          + ("" if not f1 else "   fails: " + ", ".join(
              f"{i} {a:.1f} vs {b_:.1f}" for i, a, b_ in f1)))
    print(f"  K_cond < K(x)                  : {ok2}/{n}"
          + ("" if not f2 else "   fails: " + ", ".join(
              f"{i} {a:.1f} vs {b_:.1f}" for i, a, b_ in f2)))
    if path:
        L_ = [r"\begin{table}[t]", r"\centering", r"\small",
              r"\caption{Two conditions that hold of the certified items under "
              + (shown or tag) + r", neither of them imposed. The first is the "
              r"unquestionability limit applied to the description that "
              r"certified; the second says conditioning on the prefix made it "
              r"below the cost of describing the whole sequence. The second is "
              r"sufficient for the first and not necessary, so an item may fail "
              r"it and be certified.}",
              r"\label{tab:checks-" + tag + "}",
              r"\begin{tabular}{@{}lr@{}}", r"\toprule",
              r"condition & holds \\", r"\midrule",
              rf"$\Kco < \beta \Lu + c_0 - \delta$ & {ok1} of {n} \\",
              rf"$\Kco < \hat{{K}}_\V(x)$ & {ok2} of {n} \\"]
        # BELOW THE TABULAR AND WRAPPED. As one multicolumn row it ran 350pt
        # past the margin under the 8B, which has seven exceptions where the
        # 14B has three.
        L_ += [r"\bottomrule", r"\end{tabular}"]
        if f2:
            L_.append(r"\par\vspace{2pt}\footnotesize "
                      + ("the exception: " if len(f2) == 1 else
                         f"the {len(f2)} exceptions: ")
                      + ", ".join(f"\\texttt{{{i}}} at {a:.1f} against {b_:.1f}"
                                  for i, a, b_ in f2) + ".")
        L_ += [r"\end{table}"]
        open(path, "w").write("\n".join(L_) + "\n")
        print(f"  wrote {path}")
    return ok1, ok2, n

def chain_table(items, met, machine_id, tag, path=None, shown=None):
    """How many items meet each condition, by the cell they were measured in.

    Each cell holds valid / unquestionable / certified. The three are
    nested, so the second counts only items that are also valid and the third
    only items that are also unquestionable: reading across a cell shows where
    that part of the grid loses items rather than how many reached each stage
    independently.

    The cell is the MEASURED one, rounded, from this machine's own readings.
    Validity is machine-independent, so the totals of the first number agree
    under either machine, and a cell differs only where the two measured an
    item into different cells. The other two numbers are machine-relative.

    An item this machine has not typed is placed by the other machine's
    reading. Dropping it instead made the valid column read 57 under Qwen3-8B
    against 90 under Qwen3-14B, which says the item is invalid when what is
    missing is a measurement.
    """
    # THE MACHINE TAGS. `r_hat_sd_<tag>` and `r_hat_all_<tag>` begin the same
    # way and hold a standard deviation and a list, so matching on the prefix
    # alone hands a list to round() below.
    _OTHER_TAGS = sorted({k[len("r_hat_"):] for b in items.values() for k in b
                          if k.startswith("r_hat_")
                          and not k.startswith(("r_hat_sd", "r_hat_all"))
                          and isinstance(b.get(k), (int, float))})
    g = collections.defaultdict(lambda: [0, 0, 0])
    for i, b in items.items():
        if not _is_valid(b):
            continue
        # THE CELL COMES FROM THIS MACHINE WHERE IT HAS ONE, and from the
        # other where it does not. An item with no r,t under this machine was
        # being dropped from the table altogether, so the valid column read 57
        # under Qwen3-8B against 90 under Qwen3-14B -- and validity belongs to
        # the item, not to a machine. The 33 missing were the ones the 8B has
        # not measured yet, and the table said nothing about them.
        r, t = b.get(f"r_hat_{tag}"), b.get(f"t_hat_{tag}")
        if r is None or t is None:
            for _o in _OTHER_TAGS:
                if _o == tag: continue
                r2, t2 = b.get(f"r_hat_{_o}"), b.get(f"t_hat_{_o}")
                if r2 is not None and t2 is not None:
                    r, t = r2, t2
                    break
        if r is None or t is None:
            continue
        c = (int(round(r)), int(round(t)))
        g[c][0] += 1
        if b.get(f"n_M_{tag}") is None:
            continue
        g[c][1] += 1
        if ((met.get(i) or {}).get(machine_id) or {}).get("band") not in (None, "p_lam"):
            g[c][2] += 1

    R = sorted({c[0] for c in g}) or [0]
    T = sorted({c[1] for c in g}) or [0]
    cell = lambda r, t: g.get((r, t))
    tot = [sum(v[j] for v in g.values()) for j in range(3)]

    fmt = lambda v: "/".join(str(x) for x in v) if v else "--"
    print(f"\n{tot[0]} valid, {tot[1]} unquestionable, {tot[2]} certified "
          f"under {shown or tag}")
    print(f"{'':5s}" + "".join(f"{'t'+str(t):>9s}" for t in T) + f"{'all':>9s}")
    for r in R:
        row = [sum(cell(r, t)[j] if cell(r, t) else 0 for t in T) for j in range(3)]
        print(f"r={r}  " + "".join(f"{fmt(cell(r, t)):>9s}" for t in T)
              + f"{fmt(row):>9s}")
    print(f"{'all':5s}" + "".join(
        f"{fmt([sum(cell(r, t)[j] if cell(r, t) else 0 for r in R) for j in range(3)]):>9s}"
        for t in T) + f"{fmt(tot):>9s}")

    if path:
        L = [r"\begin{table}[t]", r"\centering", r"\small",
             r"\caption{How many items meet each condition under " + (shown or tag)
             + r", by the cell they were measured in. Each entry is "
             r"\emph{valid} / \emph{unquestionable} / \emph{certified}. "
             r"The three are nested: an item counted in the second is also in "
             r"the first, and one in the third is also in the second, so the "
             r"entry reads left to right as what that part of the grid loses. "
             r"$\hat{r}$ and $\hat{t}$ are this machine's own measurements, "
             r"rounded; validity does not depend on a machine, so the first "
             r"number moves between the two tables only where the machines "
             r"measured an item into different cells. A dash is a cell with "
             r"no valid item in it.}",
             r"\label{tab:chain-" + tag + "}",
             r"\begin{tabular}{@{}l" + "r" * (len(T) + 1) + "@{}}", r"\toprule",
             r"$\hat{r}$ " + "".join(f"& $\\hat{{t}}={t}$ " for t in T)
             + r"& all \\", r"\midrule"]
        tex = lambda v: ("/".join(str(x) for x in v) if v else "--")
        for r in R:
            row = [sum(cell(r, t)[j] if cell(r, t) else 0 for t in T) for j in range(3)]
            L.append(f"{r} " + "".join(f"& {tex(cell(r, t))} " for t in T)
                     + f"& {tex(row)}" + r" \\")
        L.append(r"\midrule")
        L.append("all " + "".join(
            "& " + tex([sum(cell(r, t)[j] if cell(r, t) else 0 for r in R)
                        for j in range(3)]) + " " for t in T)
            + f"& {tex(tot)}" + r" \\")
        L += [r"\bottomrule", r"\end{tabular}", r"\end{table}"]
        open(path, "w").write("\n".join(L) + "\n")
        print(f"wrote {path}")
    return g

def heatmap(results, items, path=None, coarse_t=False, use="measured",
            title=""):
    """Accuracy over the r,t grid, one panel per examinee.

    A cell is the mean accuracy of the items measured in it, and the count is
    printed under the value, because a cell holding one item is not a cell
    holding six and the shading alone hides that. Empty cells are left blank
    rather than shaded as zero: no items measured is not the same as everything
    wrong, and colouring them would put a dark patch exactly where the battery
    is thin.

    `use` is "measured" for the r_hat and t_hat the pipeline measured, or
    "asked" for the cell the item was generated for. Measured is the honest
    one; asked is useful when comparing against the design.
    """
    import matplotlib.pyplot as plt
    import numpy as np

    key = ("r_hat", "t_hat") if use == "measured" else ("want_r", "want_t")
    lab = ("$\\hat{r}$", "$\\hat{t}$") if use == "measured" else ("r asked", "t asked")

    # keyed by item inside the cell, so n counts ITEMS. Counting rows would
    # count repetitions, and counting distinct accuracies is meaningless.
    by = collections.defaultdict(lambda: collections.defaultdict(dict))
    for r in results:
        it = items.get(r["item"]) or {}
        a, b = it.get(key[0]), it.get(key[1])
        if a is None or b is None: continue
        # Four bins, not three. The battery now reaches t = 8.6 and the old
        # top bin, 4 and above, held everything from a moderate item to the
        # deepest ones in it. On the valid set the four hold 10, 20, 13 and 4.
        tb = ((0 if b <= 1 else 1 if b <= 3 else 2 if b <= 5 else 3)
              if coarse_t else int(round(b)))
        by[r["examinee"]][(int(round(a)), tb)].setdefault(
            r["item"], []).append(ACC(r))

    names = by_capacity(by, results)

    R = sorted({k[0] for e in by for k in by[e]})
    T = sorted({k[1] for e in by for k in by[e]})
    # Wrapped over rows rather than laid out in one. Eleven panels in a row is
    # two metres of figure at a readable size, so nothing is readable at the
    # size a page allows. HEAT_ROWS gives the row lengths; the widest sets the
    # width and the rest are padded with blank axes.
    _per_row = list(HEAT_ROWS) if HEAT_ROWS else [len(names)]
    _rows, _k = [], 0
    while _k < len(names):
        _w = _per_row[min(len(_rows), len(_per_row) - 1)]
        _rows.append(names[_k:_k + _w]); _k += _w
    _wide = max(len(r) for r in _rows)
    fig, axes = plt.subplots(len(_rows), _wide,
                             figsize=(4.6 * _wide, 3.35 * len(_rows)),
                             squeeze=False)
    fig.subplots_adjust(hspace=.30)
    for _ri, _row in enumerate(_rows):
        for _ci in range(len(_row), _wide):
            axes[_ri][_ci].axis("off")
    for ax, ex in [(axes[_ri][_ci], _ex)
                   for _ri, _row in enumerate(_rows)
                   for _ci, _ex in enumerate(_row)]:
        M = np.full((len(R), len(T)), np.nan)
        N = np.zeros((len(R), len(T)), int)
        for (a, b), per_item in by[ex].items():
            M[R.index(a), T.index(b)] = np.mean(
                [np.mean(v) for v in per_item.values()])
            N[R.index(a), T.index(b)] = len(per_item)
        _cm = plt.cm.YlGn.copy()
        _cm.set_bad(alpha=0.0)
        im = ax.imshow(np.ma.masked_invalid(M), origin="lower", cmap=_cm,
                       vmin=0, vmax=1, aspect="auto", zorder=2)
        for _i in range(len(R)):
            for _j in range(len(T)):
                if np.isnan(M[_i, _j]):
                    ax.add_patch(plt.Rectangle(
                        (_j - .5, _i - .5), 1, 1, facecolor="0.93",
                        edgecolor="0.75", hatch="///", lw=.4, zorder=1))
        for i in range(len(R)):
            for j in range(len(T)):
                ax.add_patch(plt.Rectangle((j - .5, i - .5), 1, 1, fill=False,
                                           ec="0.85", lw=.8))
        _tl = (["0-1", "2-3", "4-5", "6+"] if coarse_t
           else [str(t) for t in T])
        ax.set_xticks(range(len(T)), [_tl[t] if coarse_t else str(t) for t in T])
        ax.set_yticks(range(len(R)), [str(r) for r in R])
        ax.set_xlabel(lab[1]); ax.set_ylabel(lab[0])
        # `-fallback` is a star, as in the tables, and the vendor prefix goes
        # since every panel carries it.
        _nm = ex.split("/")[-1]
        _nm = (_nm[:-len("-fallback")] + "*") if _nm.endswith("-fallback") else _nm
        ax.set_title(short(ex), fontsize=10)
        for i in range(len(R)):
            for j in range(len(T)):
                if np.isnan(M[i, j]): continue
                _dark = M[i, j] > 0.62      # readable on the deep greens
                ax.text(j, i + .16, f"{M[i,j]:.0%}", ha="center", va="center",
                        fontsize=9, color="white" if _dark else "0.15")
                ax.text(j, i - .22, f"n={N[i,j]}", ha="center", va="center",
                        fontsize=6.5, color="0.85" if _dark else "0.45")
    fig.colorbar(im, ax=axes.ravel().tolist(), fraction=.012, pad=.02,
                 label="accuracy")
    _thin = sum(1 for ex in by for c in by[ex] if len(by[ex][c]) < 3)
    _sub = (f"{_thin} cell(s) hold fewer than three items, so their shade is "
            f"one or two answers rather than a rate") if _thin else ""
    fig.suptitle((title + ("\n" if title and _sub else "") + _sub) or None,
                 fontsize=9.5 if title else 8, y=1.02,
                 color="0.2" if title else "0.35")
    if path: fig.savefig(path, dpi=150, bbox_inches="tight"); print(f"wrote {path}")
    return fig


In [ ]:
if ROWS:
    # THREE SUBSETS, each a stage of the chain, so the grid can be read at
    # every filter rather than only at the end. An item drops out at each
    # stage for a different reason, and the cells they leave behind are where
    # the battery is thin.
    #
    #   valid           the seed reproduced its own sequence on a second cold
    #                   run. Machine-independent.
    #   unquestionable  no rival inside the budget reproduces the prefix and
    #                   then diverges, under THIS machine.
    #   certified    a program the machine could run cost less than writing
    #                   the continuation out, so h is the cost of a rule.
    #
    # The valid set needs the unfiltered rows: an item questionable under this
    # machine may have been administered under the other, and ROWS has already
    # dropped it.
    _valid_ids = {i for i in ITEMS_ALL if i in _valid}
    _RV = [r for r in ROWS_ALL_RAW if r["item"] in _valid_ids]
    _unq = {i for i in ITEMS if ITEMS[i].get("n_M") is not None}
    _c = {i for i in _unq if ITEMS[i].get("band") not in (None, "p_lam")}
    _l = {i for i in _unq if ITEMS[i].get("band") == "p_lam"}

    for _tag, _rows, _ids, _what in (
            ("valid", _RV, _valid_ids, "valid"),
            ("unq", ROWS, _unq, f"valid and unquestionable under "
                                f"{MACHINE_SHOWN}"),
            ("comp", [r for r in ROWS if r["item"] in _c], _c,
             f"valid, unquestionable under {MACHINE_SHOWN} and certified "
             f"under it"),
            ("listing", [r for r in ROWS if r["item"] in _l], _l,
             f"valid, unquestionable under {MACHINE_SHOWN} and NOT "
             f"certified: h is the cost of a copy")):
        if len(_ids) < 4 or not _rows:
            continue
        heatmap(_rows, ITEMS_ALL if _tag == "valid" else ITEMS,
                coarse_t=True,
                path=f"{FIGDIR}/fig_heat_{_tag}{FIG_SUFFIX}.png",
                title=f"The {len(_ids)} items that are {_what}")
        if _tag == "unq":
            heatmap(_rows, ITEMS,
                    path=f"{FIGDIR}/fig_heat_{_tag}_fine{FIG_SUFFIX}.png",
                    title=f"The {len(_ids)} items that are {_what}, "
                          f"transformations not grouped")
else:
    print("no results file, so no heatmap")

# HOW MANY ITEMS MEET EACH CONDITION, by the cell they were measured in. The
# counts are of items and not of answers, so the marking does not enter and
# the file carries no suffix; it was written only on the prefix run, which
# meant it stopped being written at all once prefix scoring was dropped.
if True:
    chain_table(ITEMS_ALL, MET, MACHINE_ID, MACHINE_TAG,
                path=f"{FIGDIR}/tab_chain_{MACHINE_TAG}.tex",
                shown=MACHINE_SHOWN)
    # and the two conditions the certified items satisfy without being made to
    certified_checks(ITEMS_ALL, MET, MACHINE_ID, MACHINE_TAG, PRI,
                     shown=MACHINE_SHOWN,
                     path=f"{FIGDIR}/tab_checks_{MACHINE_TAG}.tex")


## 4 · Characteristic curves

Accuracy against a difficulty measure, one point per item, the mean over the
repetitions.

The fit is constrained.
An item costing nothing to continue should be answered
by anyone and one costing unboundedly much by nobody, so

$$p(x) = \frac{2}{1 + \exp\big((x/b)^k\big)}$$

which is exactly 1 at $x=0$ and falls to 0.
$b$ is where the examinee is on the
edge, in the units of the measure, so two examinees compare by one number on
the same scale as the items.
An unconstrained logistic can sit above 1 at the
origin, and does when the easy items are sparse.

In [ ]:
import re

def by_capacity(names, results=None):
    """Examinees weakest first.

    By MEASURED accuracy when the results are to hand, which works for any
    family and is what a reader wants: the columns then read left to right as
    the ladder. Sorting on the version number only worked inside one family,
    and put two open-weight models, the weakest in the set, at the end.

    Falls back to version and tier when nothing has been measured yet.
    """
    names = list(names)
    if results:
        sc = collections.defaultdict(lambda: [0, 0])
        for r in results:
            s = sc[r["examinee"]]
            s[0] += r["n_correct"]; s[1] += r["k"]
        if all(sc.get(n, [0, 0])[1] for n in names):
            return sorted(names, key=lambda n: sc[n][0] / max(sc[n][1], 1))

    def key(n):
        s = n.split("/")[-1]
        m = re.match(r"gemini-([0-9.]+)-(.*)", s)
        if not m:
            return (1, 0.0, 0, s)
        return (0, float(m.group(1)), 0 if "lite" in m.group(2) else 1, s)
    return sorted(names, key=key)

def _star(ex):
    """The short label, as the tables use. Was the model's full name, which
    made a twelve-entry legend wider than the axes it sat beside."""
    return short(ex)

def agent_curve(results, items, x="h", per_model=True, path=None,
                xlabel=None):
    """Agent characteristic curves: accuracy against a difficulty measure.

    One curve per EXAMINEE, not per item, which is why they are agent curves
    and not item characteristic curves.

    One point per item per examinee, the mean over the repetitions, so a point
    is 0, 1/3, 2/3 or 1 with three attempts. Then a curve.

    THE CURVE IS CONSTRAINED. An item that costs nothing to continue should be
    answered by anyone, and one that costs unboundedly much by nobody, so the
    fit is

        p(x) = 2 / (1 + exp((x / b) ** k))

    which is exactly 1 at x = 0 and falls to 0 as x grows, with b the scale
    (where p is 0.5) and s the WIDTH of the transition in bits. Two parameters, both
    interpretable, and neither can put the curve above 1 at the origin, which
    an unconstrained logistic can and does when the easy items are sparse.

    th is the more useful of the two: it is the difficulty at which an examinee
    is on the edge, in the units of the measure, so two examinees can be
    compared by one number on the same scale as the items.
    """
    import matplotlib.pyplot as plt
    import numpy as np
    from scipy.optimize import curve_fit

    by = collections.defaultdict(lambda: collections.defaultdict(list))
    for r in results:
        by[r["examinee"]][r["item"]].append(ACC(r))
    val = {i: items[i].get(x) for i in items}

    def f(xx, th, s):
        """A logistic in BITS: th is where the examinee is on the edge and s
        is the width of the transition, both in the units of the measure and
        free of each other.

        It was 2/(1+exp((x/b)^k)), a logistic in LOG bits, which pins the
        curve at (0,1) but ties the width to the location: a model at b = 152
        gets a transition twice as wide, in bits, as one at 76. With k also
        clamped at its lower bound for ten of twelve examinees, the curves
        could only flatten as they moved right.
        """
        return 1.0 / (1.0 + np.exp(np.clip((np.asarray(xx) - th) / s, -50, 50)))

    def fit(xs, ys):
        try:
            # k >= 2, so the curve SATURATES rather than merely passing
            # through 1. At k = 1 it leaves the origin at once and decays like
            # an exponential, which draws an examinee as already uncertain on
            # an item costing almost nothing; below 1 the slope at the origin
            # is infinite and it is worse. A plateau then a fall is what the
            # claim "an item that costs nothing is answered by anyone" looks
            # like as a shape, and it needs k > 1.
            #
            # It costs little. On this data the RMSE rises from 0.388 to 0.416
            # in the worst case and the ordering of b is unchanged, because
            # what pins k low is a handful of cheap items answered wrongly for
            # reasons that have nothing to do with cost.
            # TWICE AS MANY PSEUDO-OBSERVATIONS AT (0, 1) AS THERE ARE ITEMS. The
            # claim is that an item costing nothing is answered by anyone,
            # and as a prior it does the work the old form did by
            # construction: forcing f(0) towards 1 in a logistic requires
            # th/s to be large, so the fit shrinks s and the curve steepens.
            # On this data s falls by a fifth to a quarter, f(0) rises from
            # as low as 0.87 to 0.94 or better, th barely moves, and the
            # RMSE on the REAL points is within 0.01 of the unanchored fit.
            _n = 2 * len(xs)
            _X = np.concatenate([np.asarray(xs, float), np.zeros(_n)])
            _Y = np.concatenate([np.asarray(ys, float), np.ones(_n)])
            p, _ = curve_fit(f, _X, _Y, p0=[max(np.median(xs), 1.0), 30.0],
                             bounds=([0.1, 1.0], [1e4, 1e4]), maxfev=40000)
            return p
        except Exception:
            return None

    _lim = max(v for v in val.values() if v is not None) * 1.05
    names = by_capacity(by, results)
    cols = plt.cm.viridis(np.linspace(0, 0.85, len(names)))
    fits = {}
    if per_model:
        # wrap onto rows rather than one long strip: five panels side by side
        # are 20 inches wide and an inch and a half tall, which is unreadable
        n_p = len(names) + 1
        ncol = min(3, n_p)
        nrow = int(np.ceil(n_p / ncol))
        fig, axes = plt.subplots(nrow, ncol,
                                 figsize=(4.4 * ncol, 3.7 * nrow),
                                 sharey=True, squeeze=False)
        axes = axes.ravel()
        for k in range(n_p, len(axes)): axes[k].axis("off")
    else:
        ncol = 1
        # THE SAME SHAPE AS THE ECI SCATTER, so the two sit side by side
        # without one being taller than the other.
        fig, ax1 = plt.subplots(figsize=(8.0, 5.2)); axes = [ax1]

    for n, (ex, col) in enumerate(zip(names, cols)):
        pts = [(val[i], np.mean(v)) for i, v in by[ex].items()
               if val.get(i) is not None]
        if len(pts) < 4: continue
        xs = np.array([p[0] for p in pts]); ys = np.array([p[1] for p in pts])
        p = fit(xs, ys); fits[ex] = p
        _rmse = (float(np.sqrt(np.mean((f(xs, *p) - ys) ** 2)))
                 if p is not None else float("nan"))
        ax = axes[n] if per_model else axes[0]
        ax.scatter(xs, ys, s=26, color=col, alpha=.55, lw=0, zorder=3)
        if p is not None:
            # to the axis limit, not to this examinee's own largest item.
            # Every panel shares _lim, so drawing to xs.max() left the curve
            # stopping short of the frame and looking as though the fit ran
            # out rather than the data.
            g = np.linspace(0, _lim, 300)
            # NO LABEL when the panels are off: the summary block below
            # draws the same curves on the same axis, which put every
            # examinee in the legend twice.
            ax.plot(g, f(g, *p), color=col, lw=2, zorder=4,
                    label=(f"{_star(ex)}  $\\theta$={p[0]:.0f}")
                          if per_model else None)
        if per_model:
            ax.set_title(f"{short(ex)}   n = {len(pts)}\n"
                         + (f"half-way at {x} = {p[0]:.0f}, width "
                            f"{p[1]:.1f}, rmse {_rmse:.2f}"
                            if p is not None else "no fit"), fontsize=9)
            ax.set_xlim(-_lim * 0.03, _lim)
            ax.set_xlabel(xlabel or x)
            ax.grid(alpha=.25)
            ax.set_ylim(-0.05, 1.05)

    ax = axes[len(names)] if per_model else axes[0]
    for (ex, p), col in zip(fits.items(), cols):
        if p is None: continue
        g = np.linspace(0, _lim, 300)
        ax.plot(g, f(g, *p), color=col, lw=2.2,
                label=f"{_star(ex)}   $\\theta$={p[0]:.0f}")
    _xmax = max(v for v in val.values() if v is not None) * 1.05
    ax.set_xlim(-_xmax * 0.03, _xmax)
    ax.set_ylim(-0.05, 1.05); ax.grid(alpha=.25)
    ax.set_xlabel(xlabel or x)
    # OUTSIDE: inside it covered the dense corner of the scatter
    # INSIDE when there are no panels: twelve short labels fit the upper
    # right corner, which is empty, and an outside legend costs a fifth of
    # the width for nothing.
    ax.legend(fontsize=7.5,
              **(dict(loc="upper right") if not per_model
                 else dict(loc="upper left", bbox_to_anchor=(1.01, 1.0))),
              borderaxespad=0, frameon=False)
    if per_model:
        ax.set_title("all, fitted curves only", fontsize=9)
    for k in range(0, len(axes), ncol if per_model else 1):
        # "accuracy" alone without the panels: with one axis there is no
        # per-panel mean to distinguish it from.
        if k < len(axes):
            axes[k].set_ylabel("accuracy, mean over the reps" if per_model
                               else "accuracy")
    if not per_model:
        # WHERE THE ABILITIES ARE READ OFF. theta is the crossing at 0.5, so
        # the line says what the legend's numbers mean. It stops at 0.72 of
        # the width to clear the legend in the upper right.
        axes[0].axhline(0.5, xmax=0.72, color="0.6", ls="--", lw=1.0,
                        zorder=1)
    fig.tight_layout(rect=(0, 0, 0.80 if per_model else 1.0, 1))
    if path: fig.savefig(path, dpi=150); print(f"wrote {path}")
    return fig, fits


In [ ]:
if ROWS:
    agent_curve(ROWS, ITEMS, x="K", xlabel="$K(x_{1:n+k})$, bits",
                path=f"{FIGDIR}/fig_agent_K{FIG_SUFFIX}.png")
    # Kc dropped: it tracks Kt closely and the Kt curve on the certified
    # items, drawn below, is the one the ability numbers are read off.
else:
    print("no results file, so no curves")


## 4b · Unquestionability length against difficulty

One point per certified item.
The dashed line is the necessary condition read
as an equality,

$$N_u = \max\Big\{L_0,\ \frac{c(p) + \delta - c_0}{\beta}\Big\}$$

flat at the initial prefix for cheap items, where five terms already exclude
every rival, and rising once the rule is dear enough to need a longer one.
$\beta$ and $c_0$ come from the calibration file, for the REFERENCE machine
rather than the pooled minimum.

The x axis is unconditional $K$, and only that: the condition is stated in
terms of $c(p)$, so drawing its line against $h$ would put an asymptote on a
quantity it says nothing about.

Items sit on both sides of it, for one reason, and the pricing records confirm
it rather than leaving it to argument.
The three structural rivals all print
the prefix verbatim and then continue trivially: `cycle` repeats the list,
`hold` repeats the last term, `const` emits zero forever.
Their cost is
therefore the cost of PRINTING THE PREFIX, and the line uses a fitted average
of $\beta$ bits a term for that.

Each record stores what the cheapest rival really cost at each prefix.
At the
prefix where an item certified, the median gap between the real cost and the
predicted one is $+77$ bits for the items below the line and $-31$ for those
above, and the two move together at $\rho = -0.62$.
So a prefix that is dearer
to print than the average prices its rival out early and certifies below the
line; one that is cheaper keeps its rival affordable and certifies above.
The
rule has nothing to do with it.

In [ ]:
def cert_vs_h(items, path=None, beta=None, c0=None, L0=5, delta=22.7,
              x="K", calib=None, ref_machine="Qwen/Qwen3-8B", pricing=None,
              show_open=True, xmax=600, ymax=70, machine=""):
    """Unquestionability length against item difficulty, one point per item.

    The prediction is Proposition 4.1 read as an equality rather than a bound:

        L_u = max{ L0, (c(p) + delta - c0) / beta }

    flat at the initial prefix for cheap items, where five terms already
    exclude every rival, and rising once the rule is dear enough that a longer
    prefix is needed. Two segments and a floor.

    x is h where an item has one and K where it does not, since h exists for a
    minority; which was used is marked. beta and c0 default to the values in
    the calibration file if it is passed, and to the pooled ones otherwise.
    """
    import matplotlib.pyplot as plt
    import numpy as np

    _m = (calib or {}).get("machines", {}).get(ref_machine, {}) or {}
    _p = (calib or {}).get("pooled", {})
    beta = (_m.get("beta_listing") or _p.get("beta_listing") or 3.4) \
        if beta is None else beta
    c0 = (_m.get("c0_listing") or _p.get("c0_listing") or 135.2) \
        if c0 is None else c0

    # x is K by default: the prediction is stated in terms of c(p), K exists
    # for every certified item, and h for a minority. x="h" gives the paper's
    # version on whatever subset has one.
    _keep = [(i, b) for i, b in items.items()
             if b.get("n_M") is not None and b.get(x) is not None]
    _ids = [i for i, _ in _keep]
    # three groups, not two: an item with no h yet, one whose h came from a
    # program, and one whose h is the cost of the listing. The third is where
    # h is an upper bound rather than a measurement, so it is worth seeing
    # whether those sit anywhere in particular.
    def _grp(b):
        if b.get("h") is None: return "none"
        return "lam" if b.get("band") == "p_lam" else "prog"
    _need = lambda c: max(L0, (c + delta - c0) / beta)
    pts = [(b.get(x), b["n_M"], _grp(b), b["n_M"] - _need(b.get(x)))
           for _, b in _keep]
    if not pts:
        print("nothing certified to plot"); return

    # how many of each group reach the end. An item far above the line needed
    # a long prefix, which means rivals kept matching, which means the
    # continuation is predictable and the listing beats any rule. An item below
    # it has a rule too dear for its prefix, and the listing beats that too.
    # Both ends lose; the middle survives.
    def _rate(sel):
        q = [p for p in pts if sel(p[3]) and p[2] != "none"]
        if not q: return None
        return f"{sum(1 for p in q if p[2] == 'prog')} of {len(q)}"
    _near_ok = _rate(lambda d: 0 <= d <= 5)
    _far_ok = _rate(lambda d: d > 5)
    _below_ok = _rate(lambda d: d < 0)

    open_pts = []
    for i, b in (pricing or {}).items():
        # only items in `items`, so a filtered analysis does not draw arrows
        # for items it has excluded: the pricing file covers the whole battery
        # and `items` may be a subset of it.
        if i not in items: continue
        it = items.get(i) or {}
        if it.get("n_M") is not None: continue
        K = it.get(x) if x != "K" else (it.get("K") or b.get("K"))
        tried = (b.get("certify_at") or []) + (b.get("walk_at") or [])
        if K is None or not tried: continue
        open_pts.append((K, max(tried)))

    fig, ax = plt.subplots(figsize=(9.4, 5.4))
    for g, style, lab in (
            ("none", dict(s=34, facecolor="none", edgecolor="0.6", lw=1.0),
             "not yet annotated"),
            ("lam", dict(s=44, marker="s", facecolor="none",
                         edgecolor="#d62728", lw=1.3),
             "$Kc_{\\mathcal{M}}$ from the listing $p^\lambda$"),
            ("prog", dict(s=46, color="#1f77b4", alpha=.85, lw=0),
             "$Kc_{\\mathcal{M}}$ from a program, certified")):
        q = [p for p in pts if p[2] == g]
        if q:
            ax.scatter([p[0] for p in q], [p[1] for p in q], zorder=4,
                       label=f"{lab}   ({len(q)})", **style)

    if open_pts and show_open:
        ax.errorbar([p[0] for p in open_pts], [p[1] for p in open_pts],
                    yerr=[[0] * len(open_pts), [3.5] * len(open_pts)],
                    fmt="^", ms=5, color="0.45", lw=0, elinewidth=.9,
                    capsize=0, alpha=.75, zorder=3,
                    label=f"never certified, at least this long   "
                          f"({len(open_pts)})")

    # to xmax rather than to the largest item, so the line and the region it
    # forbids can be read past where this battery happens to stop. At K = 600
    # the condition asks for about 140 terms, which is where the argument for
    # longer sequences has to be made.
    _hi = max(xmax, max(p[0] for p in pts) * 1.05)
    g = np.linspace(0, _hi, 600)
    need = lambda c: np.maximum(L0, (np.asarray(c) + delta - c0) / beta)

    # BELOW the line is the side the condition forbids: at that prefix the
    # listing rival fits inside the budget and should have blocked the item.
    # Shaded, so the reader can see which side is which without working it out.
    ax.fill_between(g, 0, need(g), color="#d62728", alpha=.045, lw=0, zorder=1)
    # the y axis stays with the DATA, 70 terms, not with the line. At K = 600
    # the condition asks for about 140, so the line simply leaves the frame,
    # which is the point being made: the sequences are 72 long.
    ax.set_ylim(-3, ymax)
    ax.plot(g, need(g), "--", color="0.3", lw=1.4, zorder=3,
            label=fr"necessary condition, $\beta={beta:.1f}$, $c_0={c0:.0f}$")

    exc = sorted(max(0.0, p[0] - (beta * p[1] + c0 - delta)) for p in pts)
    over = [e for e in exc if e > 0]
    if over:
        band = float(np.percentile(over, 90))
        ax.plot(g, need(g - band), ":", color="#d62728", lw=1.2, zorder=3,
                label=f"and {band:.0f} bits further, which holds "
                      f"{sum(1 for e in over if e <= band)} of {len(over)}")
    ax.axhline(L0, color="0.8", lw=.8, zorder=1)

    # a straight fit through the points as well, since the prediction is a
    # claim and the data may not follow it
    xs = np.array([p[0] for p in pts], float)
    ys = np.array([p[1] for p in pts], float)
    if len(xs) > 3:
        m, c = np.polyfit(xs, ys, 1)
        r = np.corrcoef(xs, ys)[0, 1]
        ax.plot(g, m * g + c, color="#d62728", lw=1.6, zorder=5,
                label=f"least squares, $r={r:.2f}$")

    ax.set_ylabel("$L_u$, unquestionability length in terms")
    _n_over = len(over) if over else 0
    ax.set_title(f"Unquestionability length against difficulty, "
                 f"{len(pts)} of {len(pts) + (len(open_pts) if open_pts else 0)} "
                 f"valid items are unquestionable under {machine}"
                 + (f"; the other {len(open_pts)} never certified"
                    if open_pts and show_open else ""), fontsize=10)
    # the rest goes under the axis, where there is room for it
    ax.set_xlabel("$Kc_{\\mathcal{M}}$, bits" if x == "h" else "$K(x_{1:n+k})$, bits",
                  fontsize=10)
    _note = (f"shaded: the condition forbids it, and {_n_over} items sit there "
             f"by up to {max(over) if over else 0:.0f} bits.")
    if _near_ok is not None:
        _note += ("\nThe line uses a fitted average of "
                  f"{beta:.1f} bits a term for printing the prefix. Where the "
                  "real rival costs MORE than that it is ruled out early and "
                  "the item\ncertifies below the line; where it costs less it "
                  "keeps blocking and the item certifies above. At the "
                  "certifying prefix the median gap is\n+77 bits below the "
                  "line and -31 above. A program later beat the listing on "
                  + _near_ok + " of the items within five terms of the line, "
                  + _far_ok + " further above, " + _below_ok + " below.")
    fig.text(0.5, -0.02, _note, ha="center", va="top", fontsize=8,
             color="0.3")

    ax.set_xlim(-_hi * 0.02, _hi)
    ax.grid(alpha=.25)
    # OUTSIDE, as on every figure here: a legend the size of this one
    # hid whichever corner it sat in
    ax.legend(fontsize=7.5, loc="upper left", bbox_to_anchor=(1.01, 1.0),
              borderaxespad=0, frameon=False)
    fig.tight_layout(rect=(0, 0, 0.80, 1))
    if path:
        fig.savefig(path, dpi=150, bbox_inches="tight")
        print(f"wrote {path}")
    return fig


In [ ]:
if any(v.get("n_M") for v in ITEMS.values()):
    cert_vs_h(ITEMS, calib=CAL, pricing=PRI, machine=MACHINE_SHOWN, path=f"{FIGDIR}/fig_certlen_{MACHINE_TAG}.png")
    # no version against h. The condition is stated in UNCONDITIONAL K, so
    # its line and its shaded region belong on a K axis; drawn against h they
    # are an asymptote on the wrong quantity.


## 4c · Accuracy in bins

The same information as the characteristic curves, without a parametric fit.
One line per examinee with a 95% Wilson interval on its own trials, and the
mean over examinees in black.

The mean's band is an ITEM-CLUSTERED bootstrap: every examinee answers the
same items, so pooling their trials as independent would understate the
uncertainty.
Resampling items keeps the dependence.
Bin counts are printed,
because a band is only as good as what is in the bin.

In [ ]:
import re

def by_capacity(names, results=None):
    """Examinees weakest first.

    By MEASURED accuracy when the results are to hand, which works for any
    family and is what a reader wants: the columns then read left to right as
    the ladder. Sorting on the version number only worked inside one family,
    and put two open-weight models, the weakest in the set, at the end.

    Falls back to version and tier when nothing has been measured yet.
    """
    names = list(names)
    if results:
        sc = collections.defaultdict(lambda: [0, 0])
        for r in results:
            s = sc[r["examinee"]]
            s[0] += r["n_correct"]; s[1] += r["k"]
        if all(sc.get(n, [0, 0])[1] for n in names):
            return sorted(names, key=lambda n: sc[n][0] / max(sc[n][1], 1))

    def key(n):
        s = n.split("/")[-1]
        m = re.match(r"gemini-([0-9.]+)-(.*)", s)
        if not m:
            return (1, 0.0, 0, s)
        return (0, float(m.group(1)), 0 if "lite" in m.group(2) else 1, s)
    return sorted(names, key=key)

ENSEMBLE = "__all__"

def short(name):
    """A short column label. The caption says what each one is.

      gemini-3.5-flash-lite        ->  g3.5FL
      claude-haiku-4-5-20251001    ->  haiku
      claude-sonnet-5              ->  sonnet
      gpt-5-nano                   ->  gpt5na
      allenai/OLMo-2-1124-7B       ->  olmo2
      Qwen/Qwen2.5-1.5B-Instruct   ->  q1.5B
      claude-opus-5-fallback       ->  opus*

    Short enough that a table of nine examinees and a `needs' column does not
    run off the page, which is what a longer scheme did.
    """
    if name == ENSEMBLE:
        return "all"
    # "-fallback" is the same model with server-side fallbacks asked for. It
    # has to be told apart from the plain one or a table shows two rows called
    # "opus" and a reader cannot say which is which.
    _fb = name.endswith("-fallback")
    if _fb:
        return short(name[:-len("-fallback")]) + "*"
    n = name.split("/")[-1]
    m = re.match(r"gemini-([0-9.]+)-(.+)", n)
    if m:
        return "g" + m.group(1) + "".join(w[0].upper()
                                          for w in m.group(2).split("-"))
    m = re.match(r"claude-([a-z]+)", n, re.I)
    if m:
        return m.group(1).lower()
    m = re.match(r"gpt-([0-9]+)[.0-9]*-([a-z]+)", n, re.I)
    if m:
        # `sol' is a name and survives whole; everything else is cut to two,
        # which is enough to tell nano from mini.
        _suf = m.group(2).lower()
        return f"gpt{m.group(1)}{_suf if _suf == 'sol' else _suf[:2]}"
    m = re.match(r"OLMo-([0-9]+)", n, re.I)
    if m:
        return f"olmo{m.group(1)}"
    m = re.match(r"Qwen[0-9.]*-([0-9.]+B)", n, re.I)
    if m:
        return f"q{m.group(1)}"
    m = re.match(r"gemma-([0-9]+)", n, re.I)
    if m:
        return f"gemma{m.group(1)}"
    return n[:12]

# short() and ENSEMBLE are defined here as well as in the tables cell,
# so the figures do not depend on the order the cells are run in.

def _wilson(k, n, z=1.96):
    """95% Wilson interval. Sensible at 0 and 1, where the normal one is not."""
    if n == 0: return (0.0, 1.0)
    p = k / n
    d = 1 + z * z / n
    c = (p + z * z / (2 * n)) / d
    h = z * math.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / d
    return (max(0.0, c - h), min(1.0, c + h))

def kt_histogram(items, ids, path=None, title=None, bins=16, xlabel=None):
    """How the certified items are spread over the difficulty scale.

    The correlation tables say how well a measure orders items; this says
    what there is to order. A battery bunched at one end reports a
    correlation that rests on a handful of items at the other, so the shape
    is worth showing beside the number.

    Bars are counts; the rug beneath marks every item, because a bar of
    height one and a bar of height six read alike at a glance.
    """
    xs = [items[i]["Kt"] for i in ids
          if items.get(i) and items[i].get("Kt") is not None]
    if len(xs) < 4:
        print("  too few items for a histogram"); return
    xs = np.asarray(xs, float)
    fig, ax = plt.subplots(figsize=(8.0, 5.2))
    ax.hist(xs, bins=bins, color="#4c72b0", alpha=.85,
            edgecolor="white", lw=.8)
    ax.plot(xs, np.full(len(xs), -0.18), "|", color="0.35", ms=7,
            clip_on=False)
    ax.axvline(float(np.median(xs)), color="0.35", ls="--", lw=1.0)
    ax.annotate(f"median {np.median(xs):.0f}",
                (float(np.median(xs)), ax.get_ylim()[1]),
                xytext=(4, -10), textcoords="offset points",
                fontsize=8.5, color="0.35", va="top")
    ax.set_xlabel(xlabel or
                  r"$\widehat{Kt}_\mathcal{M}(x_{n+1:n+k} \mid x_{1:n})$, bits")
    ax.set_ylabel("items")
    ax.set_ylim(bottom=0)
    ax.grid(axis="y", color="0.92", lw=.8)
    ax.set_axisbelow(True)
    if title != "":
        ax.set_title(title or
                     f"{len(xs)} certified items, {xs.min():.0f} to "
                     f"{xs.max():.0f} bits", fontsize=9.5)
    fig.tight_layout()
    if path:
        fig.savefig(path, dpi=150, bbox_inches="tight"); print(f"wrote {path}")
    plt.close(fig)

def acc_binned(results, items, x="K", bins="quantile", path=None,
               xlabel=None,
              boots=2000, seed=0, title=None):
    """Accuracy in bins of a difficulty measure, with bands.

    One line per examinee with a 95% Wilson interval on its own trials, and a
    black line for the mean over examinees.

    The mean's interval is an ITEM-CLUSTERED bootstrap: every examinee answers
    the same items, so treating their trials as independent would understate
    the uncertainty. Resampling items rather than trials keeps the dependence.

    Bin counts are printed along the top, because a band is only as good as
    what is in the bin and a bin holding three items should be read as such.
    """
    import matplotlib.pyplot as plt
    import numpy as np
    rng = np.random.default_rng(seed)

    val = {i: items[i].get(x) for i in items}
    rows = [r for r in results if val.get(r["item"]) is not None]
    if not rows:
        print(f"no items carry {x}"); return

    xs = sorted({val[r["item"]] for r in rows})
    # "quantile" puts the same number of items in each bin, which keeps every
    # band narrow but hides where on the scale the items sit. "equal" cuts the
    # RANGE evenly, which is what shows whether accuracy really approaches 1
    # near zero and 0 at the top; the price is that some bins hold few items,
    # and the counts printed on the figure say which.
    if isinstance(bins, str):
        nb = 4 if len(xs) < 30 else 5
        if bins == "equal":
            bins = list(np.linspace(0, max(xs), nb + 1))
        elif bins == "equal_log":
            bins = [0.0] + list(np.geomspace(max(min(xs), 1.0), max(xs), nb))
        else:
            bins = list(dict.fromkeys(
                float(v) for v in np.quantile(xs, np.linspace(0, 1, nb + 1))))
        bins[0] -= 1e-9; bins[-1] += 1e-9
    elif bins is None:
        q = np.quantile(xs, [0, .25, .5, .75, 1.0])
        bins = list(dict.fromkeys([float(v) for v in q]))
        bins[0] -= 1e-9; bins[-1] += 1e-9
    lab = [f"{bins[i]:.0f}\u2013{bins[i+1]:.0f}" for i in range(len(bins) - 1)]

    def which(v):
        for i in range(len(bins) - 1):
            if bins[i] <= v <= bins[i + 1]: return i
        return None

    names = by_capacity({r["examinee"] for r in rows}, rows)
    cols = plt.cm.viridis(np.linspace(0, .82, len(names)))
    fig, ax = plt.subplots(figsize=(9.6, 4.6))
    centres = np.arange(len(lab))

    # per examinee: successes and trials in each bin, Wilson on those
    for ex, col in zip(names, cols):
        ks, ns = np.zeros(len(lab)), np.zeros(len(lab))
        for r in rows:
            if r["examinee"] != ex: continue
            b = which(val[r["item"]])
            if b is None: continue
            ks[b] += r["n_correct"]; ns[b] += r["k"]
        p = np.where(ns > 0, ks / np.maximum(ns, 1), np.nan)
        lo = np.array([_wilson(ks[i], ns[i])[0] for i in range(len(lab))])
        hi = np.array([_wilson(ks[i], ns[i])[1] for i in range(len(lab))])
        ax.plot(centres, p, "-o", color=col, ms=5, lw=1.8,
                label=short(ex), zorder=4)
        ax.fill_between(centres, lo, hi, color=col, alpha=.16, lw=0, zorder=2)

    # the mean over examinees, with an item-clustered bootstrap
    per_item = collections.defaultdict(list)
    for r in rows: per_item[r["item"]].append(r)
    ids = list(per_item)
    mean_p, mean_lo, mean_hi, counts = [], [], [], []
    for b in range(len(lab)):
        here = [i for i in ids if which(val[i]) == b]
        counts.append(len(here))
        if not here:
            mean_p.append(np.nan); mean_lo.append(np.nan); mean_hi.append(np.nan)
            continue
        f = lambda sample: np.mean([np.mean([ACC(rr)
                                             for rr in per_item[i]])
                                    for i in sample])
        mean_p.append(f(here))
        draws = [f(list(rng.choice(here, len(here), replace=True)))
                 for _ in range(boots)]
        mean_lo.append(float(np.percentile(draws, 2.5)))
        mean_hi.append(float(np.percentile(draws, 97.5)))
    ax.plot(centres, mean_p, "-s", color="black", ms=5, lw=2.2,
            label="mean over examinees", zorder=6)
    ax.fill_between(centres, mean_lo, mean_hi, color="black", alpha=.12,
                    lw=0, zorder=3)

    # the count goes UNDER the bin it belongs to, in the tick label, so a
    # reader cannot take a band seriously without seeing what is behind it
    ax.set_xticks(centres, [f"{l}\nn={n}" for l, n in zip(lab, counts)])
    ax.set_xlabel(xlabel or f"{x}, bits")
    ax.set_ylabel("accuracy")
    ax.set_ylim(-0.03, 1.04); ax.set_xlim(-0.55, len(lab) - 0.45)
    ax.grid(alpha=.25, axis="y")
    ax.set_title(title or (f"Accuracy in bins of {x}, {len(ids)} items.  "
                           f"bands: 95% Wilson per examinee,\n"
                           f"item-clustered bootstrap for the mean"),
                 fontsize=9.5)
    ax.legend(fontsize=7.5, loc="upper left", bbox_to_anchor=(1.01, 1.0),
              borderaxespad=0, frameon=False)
    fig.tight_layout(rect=(0, 0, 0.80, 1))
    if path: fig.savefig(path, dpi=150); print(f"wrote {path}")
    return fig, dict(bins=bins, counts=counts, mean=mean_p)


In [ ]:
if ROWS:
    for _x, _lab in (("K", "$K(x_{1:n+k})$, bits"),
                     ("Kt", r"$\widehat{Kt}_\mathcal{M}(x_{n+1:n+k} \mid "
                            r"x_{1:n})$, bits")):
        for _b in ("equal", "quantile"):
            acc_binned(ROWS, ITEMS, x=_x, bins=_b, xlabel=_lab,
                       path=f"{FIGDIR}/fig_bin_{_x}_{_b}{FIG_SUFFIX}.png")

else:
    print("no results file, so no binned curves")


### The same two, on the compressible items only

On the rest the machine
fell back on writing the answer out, so their $h$ is the cost of the listing
and not of any rule: an upper bound rather than a measurement, and one that
moves with how expensive the terms are to print rather than with the item.

Restricting to the twenty is not cherry-picking a better answer.
It is
dropping the items whose $h$ was never measured, and the difference is large:
$h$ against accuracy goes from $-0.65$ over all forty-nine to $-0.74$ over
these twenty.

In [ ]:
_comp = [i for i in _hh if ITEMS[i].get("band") != "p_lam"]
_ROWS_C = [r for r in ROWS if r["item"] in set(_comp)]
print(f"{len(_comp)} certified items, {len(_ROWS_C)} result rows")

if len(_comp) > 5 and _ROWS_C:
    agent_curve(_ROWS_C, ITEMS, x="Kt",
                xlabel=r"$\widehat{Kt}_\mathcal{M}(x_{n+1:n+k} \mid x_{1:n})$, "
                       r"bits, certified only",
                path=f"{FIGDIR}/fig_agent_Kt_comp{FIG_SUFFIX}.png")
    # THE SAME FIT WITHOUT THE PANELS. The grid shows how well each curve
    # holds its points; this shows the curves against each other, which is
    # what an ability ordering is, and fits a column where the grid does not.
    agent_curve(_ROWS_C, ITEMS, x="Kt", per_model=False,
                xlabel=r"$\widehat{Kt}_\mathcal{M}(x_{n+1:n+k} \mid x_{1:n})$, "
                       r"bits, certified only",
                path=f"{FIGDIR}/fig_agent_Kt_curves{FIG_SUFFIX}.png")
    agent_curve(_ROWS_C, ITEMS, x="K", xlabel="$K(x_{1:n+k})$, bits, "
                                             "certified only",
                path=f"{FIGDIR}/fig_agent_K_comp{FIG_SUFFIX}.png")
    # the same grid on the certified items alone
    for _x, _lab in (("K", "$K(x_{1:n+k})$, bits, certified only"),
                     ("Kt", r"$\widehat{Kt}_\mathcal{M}(x_{n+1:n+k} \mid "
                            r"x_{1:n})$, bits, certified only")):
        for _b in ("equal", "quantile"):
            acc_binned(_ROWS_C, ITEMS, x=_x, bins=_b, xlabel=_lab,
                       path=f"{FIGDIR}/fig_bin_{_x}_comp_{_b}{FIG_SUFFIX}.png")
    # WHAT THERE IS TO ORDER, beside how well it is ordered.
    kt_histogram(ITEMS, _comp,
                 path=f"{FIGDIR}/fig_bin_Kt_histogram{FIG_SUFFIX}.png")
else:
    print("too few certified items to draw")


## 4d · What an answer cost the examinee

Two kinds of number on one figure.
The x axis is a description length, bits
under the reference machine.
The y axis is output tokens the EXAMINEE spent,
thinking included: work, not code.

A benchmark that reports only accuracy hides the second, and two models scoring
alike while one spends ten times the tokens are not alike.

Runs made before the evaluator recorded tokens have none, and those examinees
are NAMED on the figure rather than drawn as zero, so a gap reads as a gap.

In [ ]:
def short(name):
    """A short column label. The caption says what each one is.

      gemini-3.5-flash-lite        ->  g3.5FL
      claude-haiku-4-5-20251001    ->  haiku
      claude-sonnet-5              ->  sonnet
      gpt-5-nano                   ->  gpt5na
      allenai/OLMo-2-1124-7B       ->  olmo2
      Qwen/Qwen2.5-1.5B-Instruct   ->  q1.5B
      claude-opus-5-fallback       ->  opus*

    Short enough that a table of nine examinees and a `needs' column does not
    run off the page, which is what a longer scheme did.
    """
    if name == ENSEMBLE:
        return "all"
    # "-fallback" is the same model with server-side fallbacks asked for. It
    # has to be told apart from the plain one or a table shows two rows called
    # "opus" and a reader cannot say which is which.
    if name.endswith("-fallback"):
        return short(name[:-len("-fallback")]) + "*"
    n = name.split("/")[-1]
    m = re.match(r"gemini-([0-9.]+)-(.+)", n)
    if m:
        return "g" + m.group(1) + "".join(w[0].upper()
                                          for w in m.group(2).split("-"))
    m = re.match(r"claude-([a-z]+)", n, re.I)
    if m:
        return m.group(1).lower()
    m = re.match(r"gpt-([0-9]+)[.0-9]*-([a-z]+)", n, re.I)
    if m:
        # `sol' is a name and survives whole; everything else is cut to two,
        # which is enough to tell nano from mini.
        _suf = m.group(2).lower()
        return f"gpt{m.group(1)}{_suf if _suf == 'sol' else _suf[:2]}"
    m = re.match(r"OLMo-([0-9]+)", n, re.I)
    if m:
        return f"olmo{m.group(1)}"
    m = re.match(r"Qwen[0-9.]*-([0-9.]+B)", n, re.I)
    if m:
        return f"q{m.group(1)}"
    m = re.match(r"gemma-([0-9]+)", n, re.I)
    if m:
        return f"gemma{m.group(1)}"
    return n[:12]

import re
import collections

def by_capacity(names, results=None):
    """Examinees weakest first.

    By MEASURED accuracy when the results are to hand, which works for any
    family and is what a reader wants: the columns then read left to right as
    the ladder. Sorting on the version number only worked inside one family,
    and put two open-weight models, the weakest in the set, at the end.

    Falls back to version and tier when nothing has been measured yet.
    """
    names = list(names)
    if results:
        sc = collections.defaultdict(lambda: [0, 0])
        for r in results:
            s = sc[r["examinee"]]
            s[0] += r["n_correct"]; s[1] += r["k"]
        if all(sc.get(n, [0, 0])[1] for n in names):
            return sorted(names, key=lambda n: sc[n][0] / max(sc[n][1], 1))

    def key(n):
        s = n.split("/")[-1]
        m = re.match(r"gemini-([0-9.]+)-(.*)", s)
        if not m:
            return (1, 0.0, 0, s)
        return (0, float(m.group(1)), 0 if "lite" in m.group(2) else 1, s)
    return sorted(names, key=key)

def tokens_plot(results, items, path=None, x="h"):
    """What each answer COST the examinee, against the cost of the item.

    Two different kinds of number sit on this figure. The x axis is a
    description length, bits under the reference machine. The y axis is output
    tokens the EXAMINEE spent, thinking included: work, not code. A benchmark
    that only reports accuracy hides the second, and two models scoring alike
    while one spends ten times the tokens are not alike.

    Rows without `tok_out` are from a run made before the evaluator recorded
    it. They are NOT plotted as zero, and the examinees they belong to are
    named on the figure, so a gap reads as a gap.
    """
    import matplotlib.pyplot as plt
    import numpy as np

    have, missing = collections.defaultdict(list), []
    for ex in by_capacity({r["examinee"] for r in results}, results):
        rows = [r for r in results if r["examinee"] == ex]
        if not any(r.get("tok_out") for r in rows):
            missing.append((ex, len(rows))); continue
        for r in rows:
            it = items.get(r["item"], {})
            if x == "Kt":
                # as the correlation tables read it: the conditional cost plus
                # the log of the work, from the program that certified
                _kc, _t = it.get("K_cond"), it.get("tau")
                v = (_kc + math.log2(_t)) if (_kc is not None and _t) else None
            else:
                v = it.get(x)
            if v is None or not r.get("tok_out"): continue
            have[ex].append((v, r["tok_out"], ACC(r),
                             r.get("think_share")))

    fig, ax = plt.subplots(figsize=(9.8, 4.8))
    if have:
        cols = plt.cm.viridis(np.linspace(0, .82, len(have)))
        for (ex, pts), c in zip(have.items(), cols):
            xs = np.array([p[0] for p in pts]); ys = np.array([p[1] for p in pts])
            ok = np.array([p[2] for p in pts])
            # ONE ENTRY PER EXAMINEE, and the filled-or-hollow convention
            # explained once. Two entries each put thirty-two labels over the
            # data and the legend covered most of the points.
            ax.scatter(xs[ok >= .999], ys[ok >= .999], s=30, color=c, lw=0,
                       alpha=.75, label=short(ex))
            ax.scatter(xs[ok < .999], ys[ok < .999], s=34, facecolor="none",
                       edgecolor=c, lw=1.1, alpha=.8)
        ax.set_yscale("log")
        ax.set_ylabel("output tokens spent, thinking included")
        # OUTSIDE THE AXES. Inside, it sat on the cheap-and-quick corner where
        # most of the answers are.
        from matplotlib.lines import Line2D
        _h, _l = ax.get_legend_handles_labels()
        _h += [Line2D([], [], ls="", marker="o", color="0.35", ms=5),
               Line2D([], [], ls="", marker="o", mfc="none", mec="0.35", ms=5)]
        _l += ["all five right", "not all right"]
        ax.legend(_h, _l, fontsize=7, loc="upper left",
                  bbox_to_anchor=(1.01, 1.0), borderaxespad=0, frameon=False)
    else:
        ax.set_yticks([])

    ax.set_xlabel({"h": r"$Kc_\mathcal{M}$, bits",
                   "K": r"$\hat{K}_\mathcal{V}$, bits",
                   "Kt": r"$\widehat{Kt}_\mathcal{M}$, bits"}.get(x, x))
    ax.grid(alpha=.25)
    if missing:
        msg = ("NO TOKEN DATA for "
               + ", ".join(f"{short(e)} ({n} rows)" for e, n in missing)
               + ".\nThose runs predate the evaluator recording it. Re-run "
                 "them to fill this in;\nnothing here can be recovered from "
                 "the answers alone.")
        ax.text(.5, .5 if not have else .04, msg, transform=ax.transAxes,
                ha="center", va="center" if not have else "bottom",
                fontsize=9 if not have else 7.5,
                color="#d62728" if not have else "0.4",
                bbox=dict(fc="white", ec="#d62728" if not have else "0.8",
                          alpha=.9))
    ax.set_title(f"What an answer cost the examinee, against what the item "
                 f"costs\n{len(have)} examinee(s) with token data, "
                 f"{len(missing)} without", fontsize=9.5)
    fig.tight_layout(rect=(0, 0, 0.80, 1))
    if path: fig.savefig(path, dpi=150); print(f"wrote {path}")
    return fig


In [ ]:
if ROWS:
    tokens_plot(ROWS, ITEMS, x="K", path=f"{FIGDIR}/fig_tokens_K{FIG_SUFFIX}.png")
    if _hh:
        tokens_plot(ROWS, ITEMS, x="h", path=f"{FIGDIR}/fig_tokens_h{FIG_SUFFIX}.png")
tokens_plot(ROWS, ITEMS, x="Kt", path=f"{FIGDIR}/fig_tokens_Kt{FIG_SUFFIX}.png")


## 5 · Correlations

Spearman, because $K$ and $h$ are bits and accuracy is bounded at 0 and 1, so neither is linear in the other.

The second column is what the measure COSTS to obtain. A measure that predicts well and costs a night is not the same finding as one that predicts well and costs nothing.

In [ ]:
import re

def by_capacity(names, results=None):
    """Examinees weakest first.

    By MEASURED accuracy when the results are to hand, which works for any
    family and is what a reader wants: the columns then read left to right as
    the ladder. Sorting on the version number only worked inside one family,
    and put two open-weight models, the weakest in the set, at the end.

    Falls back to version and tier when nothing has been measured yet.
    """
    names = list(names)
    if results:
        sc = collections.defaultdict(lambda: [0, 0])
        for r in results:
            s = sc[r["examinee"]]
            s[0] += r["n_correct"]; s[1] += r["k"]
        if all(sc.get(n, [0, 0])[1] for n in names):
            return sorted(names, key=lambda n: sc[n][0] / max(sc[n][1], 1))

    def key(n):
        s = n.split("/")[-1]
        m = re.match(r"gemini-([0-9.]+)-(.*)", s)
        if not m:
            return (1, 0.0, 0, s)
        return (0, float(m.group(1)), 0 if "lite" in m.group(2) else 1, s)
    return sorted(names, key=key)

def _rank(v):
    idx = sorted(range(len(v)), key=lambda i: v[i]); r = [0.0] * len(v); i = 0
    while i < len(idx):
        j = i
        while j + 1 < len(idx) and v[idx[j + 1]] == v[idx[i]]: j += 1
        a = (i + j) / 2 + 1
        for k in range(i, j + 1): r[idx[k]] = a
        i = j + 1
    return r

def spearman(x, y):
    """Rank correlation, and the n it was computed on.

    Spearman rather than Pearson because K and h are bits, which are a log
    scale already, and accuracy is bounded at 0 and 1. Neither is linear in the
    other and there is no reason to expect them to be.
    """
    p = [(a, b) for a, b in zip(x, y) if a is not None and b is not None]
    if len(p) < 4: return None, len(p)
    rx, ry = _rank([a for a, _ in p]), _rank([b for _, b in p])
    n = len(p); mx, my = sum(rx) / n, sum(ry) / n
    dx = math.sqrt(sum((a - mx) ** 2 for a in rx))
    dy = math.sqrt(sum((b - my) ** 2 for b in ry))
    return (sum((a - mx) * (b - my) for a, b in zip(rx, ry)) / (dx * dy)
            if dx * dy else 0.0), n

def short(name):
    """A short column label. The caption says what each one is.

      gemini-3.5-flash-lite        ->  g3.5FL
      claude-haiku-4-5-20251001    ->  haiku
      claude-sonnet-5              ->  sonnet
      gpt-5-nano                   ->  gpt5na
      allenai/OLMo-2-1124-7B       ->  olmo2
      Qwen/Qwen2.5-1.5B-Instruct   ->  q1.5B
      claude-opus-5-fallback       ->  opus*

    Short enough that a table of nine examinees and a `needs' column does not
    run off the page, which is what a longer scheme did.
    """
    if name == ENSEMBLE:
        return "all"
    # "-fallback" is the same model with server-side fallbacks asked for. It
    # has to be told apart from the plain one or a table shows two rows called
    # "opus" and a reader cannot say which is which.
    if name.endswith("-fallback"):
        return short(name[:-len("-fallback")]) + "*"
    n = name.split("/")[-1]
    m = re.match(r"gemini-([0-9.]+)-(.+)", n)
    if m:
        return "g" + m.group(1) + "".join(w[0].upper()
                                          for w in m.group(2).split("-"))
    m = re.match(r"claude-([a-z]+)", n, re.I)
    if m:
        return m.group(1).lower()
    m = re.match(r"gpt-([0-9]+)[.0-9]*-([a-z]+)", n, re.I)
    if m:
        # `sol' is a name and survives whole; everything else is cut to two,
        # which is enough to tell nano from mini.
        _suf = m.group(2).lower()
        return f"gpt{m.group(1)}{_suf if _suf == 'sol' else _suf[:2]}"
    m = re.match(r"OLMo-([0-9]+)", n, re.I)
    if m:
        return f"olmo{m.group(1)}"
    m = re.match(r"Qwen[0-9.]*-([0-9.]+B)", n, re.I)
    if m:
        return f"q{m.group(1)}"
    m = re.match(r"gemma-([0-9]+)", n, re.I)
    if m:
        return f"gemma{m.group(1)}"
    return n[:12]

def _terms_of(row):
    """The answer as a list of terms, for comparing one attempt with another.

    Compared after parsing rather than as raw text, because "10 metres" and
    "10m" are the same answer and the scoring judge already treats them so.
    """
    a = (row.get("answer") or "").strip().lower()
    if not a:
        return None                      # a refusal or a cut reply says nothing
    return [t.strip() for t in a.split(",") if t.strip()]

def _common_prefix(seqs):
    """How many leading terms every one of them shares."""
    if not seqs or any(s is None for s in seqs):
        return None
    n = 0
    for parts in zip(*seqs):
        if len(set(parts)) > 1:
            break
        n += 1
    return n

def score_kinds(results):
    """Per examinee and item: how much was right, and how much was the same.

    TWO GROUPS, and they answer different questions.

    prefix% is the fraction of the k terms given in order before the first
    mistake, so an attempt scores between 0 and 1. full is that attempt scored
    1 or 0: every term right, or not.

    Within each group, worst, mean and best run over the three attempts. The
    gap between worst and best is how much of a score is luck.

    agree is not about the scores at all. It is whether the three attempts said
    the SAME THING, right or wrong. An examinee that answers "a, b, x, y, z"
    three times running is deterministic and wrong; one that gives three
    different wrong answers is guessing, and the two score alike. For prefix%
    it is how far the three coincide, as a fraction of k; for full, whether all
    three are identical end to end.

    An attempt with no answer at all is left out of the agreement, since three
    refusals agree perfectly and that is not the sense meant here.
    """
    by = collections.defaultdict(lambda: collections.defaultdict(list))
    for r in results:
        by[r["examinee"]][r["item"]].append(r)
    out = {}
    for e, d in by.items():
        per = {}
        for i, rows in d.items():
            # not ACC: this table shows both markings side by side, so the
            # fractions are needed whatever the run is ranking on
            frac = [r["n_correct"] / r["k"] for r in rows]
            perfect = [1.0 if f >= 0.999 else 0.0 for f in frac]
            said = [_terms_of(r) for r in rows]
            spoke = [s for s in said if s is not None]
            k = rows[0]["k"]
            cp = _common_prefix(spoke) if len(spoke) > 1 else None
            # an examinee may return more terms than were asked for, and the
            # shared prefix is then longer than k, which showed as agreement
            # over 100 per cent
            if cp is not None: cp = min(cp, k)
            per[i] = dict(
                worst=min(frac), best=max(frac), mean=st.mean(frac),
                f_worst=min(perfect), f_best=max(perfect),
                f_mean=st.mean(perfect),
                p_agree=(cp / k) if cp is not None else None,
                f_agree=(1.0 if len(spoke) > 1 and len(set(map(tuple, spoke))) == 1
                         else 0.0) if len(spoke) > 1 else None,
                reps=len(rows), spoke=len(spoke))
        out[e] = per
    return out
def accuracy_by_item(results):
    """examinee -> item -> mean accuracy over the repetitions."""
    by = collections.defaultdict(lambda: collections.defaultdict(list))
    for r in results:
        by[r["examinee"]][r["item"]].append(ACC(r))
    return {e: {i: st.mean(v) for i, v in d.items()} for e, d in by.items()}

def pairwise_matrix(results, items, path=None, extra=("Kt", "h", "K"),
                    title=None):
    """How much every pair of examinees agrees, item by item, with the
    measures alongside them as further columns.

    An examinee is a vector over the items: what it scored on each. Two
    examinees that find the same items hard are correlated whatever their
    level, so this says who behaves alike rather than who is better. The
    measures are vectors over the same items with the sign flipped, since a
    dear item is a hard one, and they join the matrix as if they were
    examinees: a measure that predicts well correlates with the examinees
    about as strongly as they correlate with each other, and that comparison
    is the point of putting them in the same matrix.

    Spearman throughout, over the items both members of the pair answered.
    """
    acc = accuracy_by_item(results)
    names = by_capacity(acc, results)
    cols = {e: acc[e] for e in names}
    # "h" is the field name the record has carried since before the
    # quantities were named; the label is what the paper calls it.
    _lab = {"Kt": r"$Kt_\mathcal{M}$", "h": r"$Kc_\mathcal{M}$",
            "K": r"$K_\mathcal{V}$",
            "K_cond": r"$K_\mathcal{M}$", "tau": r"$\tau_\mathcal{M}$"}
    for f in extra:
        v = {i: -items[i][f] for i in items
             if items.get(i, {}).get(f) is not None}
        if len(v) >= 4:
            cols[_lab.get(f, f)] = v
            names = names + [_lab.get(f, f)]

    n = len(names)
    M = np.full((n, n), np.nan)
    for a in range(n):
        for b in range(n):
            ia, ib = cols[names[a]], cols[names[b]]
            sh = sorted(set(ia) & set(ib))
            if len(sh) >= 4:
                # spearman returns (rho, n), not a float
                _r, _n = spearman([ia[i] for i in sh], [ib[i] for i in sh])
                if _r is not None:
                    M[a, b] = _r

    short_ = [e if e.startswith("$") else short(e) for e in names]
    fig, ax = plt.subplots(figsize=(0.52 * n + 3.2, 0.52 * n + 2.6))
    im = ax.imshow(np.ma.masked_invalid(M), cmap="RdYlGn", vmin=-1, vmax=1)
    ax.set_xticks(range(n)); ax.set_xticklabels(short_, rotation=90, fontsize=7)
    ax.set_yticks(range(n)); ax.set_yticklabels(short_, fontsize=7)
    # the measures are set off from the examinees by a rule, since they are a
    # different kind of thing sharing the axis
    _k = len(names) - sum(1 for e in names if e.startswith("$"))
    for _p in (_k - .5,):
        ax.axhline(_p, color="0.25", lw=1.1); ax.axvline(_p, color="0.25", lw=1.1)
    for a in range(n):
        for b in range(n):
            if np.isnan(M[a, b]): continue
            ax.text(b, a, f"{M[a, b]:.2f}".replace("0.", ".").replace("-", "\u2212"),
                    ha="center", va="center", fontsize=5.6,
                    color="white" if abs(M[a, b]) > .65 else "0.15")
    fig.colorbar(im, ax=ax, fraction=.035, pad=.02, label="Spearman")
    ax.set_title(title or "", fontsize=9)
    fig.tight_layout()
    if path:
        fig.savefig(path, dpi=150, bbox_inches="tight")
        print(f"wrote {path}")
    return fig
ENSEMBLE = "__all__"        # the column name for the mean over examinees

def corr_table(results, items, rows, title, note="", only=None, kind="mean",
               ensemble=True):
    """A measure per row, an examinee per column, Spearman against accuracy.

    `rows` is (label, what it needs, function of the item record). The second
    column is there because a measure that predicts well and costs a night is
    not the same finding as one that predicts well and costs nothing.
    """
    # kind is "worst", "best" or "mean" over the repetitions
    _sk = score_kinds(results)
    acc = {e: {i: v[kind] for i, v in d.items()} for e, d in _sk.items()}
    names = by_capacity(acc, results)

    # A MODEL THAT HAS A FALLBACK APPEARS TWICE, once as itself and once as
    # `-fallback`, and the two are near-copies: the fallback run differs only
    # on the items where the model declined. Keeping both puts that model in
    # the ensemble twice and makes the `all` column a weighted average with a
    # weight nobody chose. So where a `-fallback` twin exists, the bare name
    # goes and the fallback stays, since it is the one with the wider coverage.
    _fb = {e[:-len("-fallback")] for e in names if e.endswith("-fallback")}
    if _fb:
        names = [e for e in names if e not in _fb]
    if ensemble and len(names) > 1:
        _all = collections.defaultdict(list)
        for e in names:
            for i, v in acc[e].items(): _all[i].append(v)
        acc = dict(acc)
        acc[ENSEMBLE] = {i: st.mean(v) for i, v in _all.items()}
        names = names + [ENSEMBLE]
    common = sorted(set.intersection(*[set(acc[e]) for e in names]))
    if only is not None:
        common = [i for i in common if i in set(only)]
    out = []
    for label, needs, f in rows:
        cells = []
        for e in names:
            a = [acc[e][i] for i in common]
            v = [f(items[i]) if i in items else None for i in common]
            s, n = spearman(v, a)
            cells.append((s, n))
        out.append((label, needs, cells))
    # the n actually used, not the number of items offered: a measure missing
    # on some items is correlated on the rest, and reporting the larger number
    # would overstate every row that has gaps
    ns = [n for _, _, cells in out for _, n in cells if n]
    return dict(title=title, note=note, names=names, rows=out, kind=kind,
                n_items=max(ns) if ns else 0,
                n_min=min(ns) if ns else 0)

def show_corr(tab):
    print(f"\n{tab['title']}   [{tab['kind']}@3]   n = {tab['n_items']} items"
          + (f" (as few as {tab['n_min']} where a measure is missing)"
             if tab["n_min"] != tab["n_items"] else ""))
    if tab["note"]: print(f"  {tab['note']}")
    # no `needs' column. It said what each measure costs to obtain, which is
    # worth knowing once in the prose and not in every table: with nine
    # examinees the table ran off the page.
    w = max(len(r[0]) for r in tab["rows"]) + 2
    print(f"\n{'':{w}s}"
          + "".join(f"{short(e)+'@'+tab['kind'] if e == ENSEMBLE else short(e):>11s}"
                    for e in tab["names"]))
    for label, needs, cells in tab["rows"]:
        line = f"{label:{w}s}"
        for s, n in cells:
            line += f"{s:+11.2f}" if s is not None else f"{'-':>11s}"
        print(line)

def scores_table(results, path=None, only=None, note=""):
    """What the examinees scored, in two groups of four.

    prefix% is the fraction of the k terms given in order before the first
    mistake; full is that same attempt scored all or nothing. Within each,
    worst, mean and best run over the three attempts, and agree says how much
    the three attempts SAID THE SAME, right or wrong.
    """
    if only is not None:
        results = [r for r in results if r["item"] in set(only)]
    sk = score_kinds(results)
    out = []
    for e in by_capacity(sk, results):
        d = sk[e]
        g = lambda k: [v[k] for v in d.values() if v[k] is not None]
        m = lambda k: st.mean(g(k)) if g(k) else None
        out.append(dict(
            examinee=e, short=short(e), n=len(d),
            worst=m("worst"), mean=m("mean"), best=m("best"), pagree=m("p_agree"),
            fworst=m("f_worst"), fmean=m("f_mean"), fbest=m("f_best"),
            fagree=m("f_agree")))
    pc = lambda v: f"{v:6.1%}" if v is not None else f"{'-':>6s}"
    print(f"{'':24s} {'prefix%':^30s}  {'full':^30s}")
    print(f"{'examinee':24s} {'worst':>7s}{'mean':>7s}{'best':>7s}{'agree':>8s}  "
          f"{'worst':>7s}{'mean':>7s}{'best':>7s}{'agree':>8s}")
    for r in out:
        print(f"{r['short']:24s} {pc(r['worst'])} {pc(r['mean'])} {pc(r['best'])} "
              f"{pc(r['pagree'])}  {pc(r['fworst'])} {pc(r['fmean'])} "
              f"{pc(r['fbest'])} {pc(r['fagree'])}")
    if path:
        p = lambda v: (f"{v:.1%}".replace("%", r"\%") if v is not None else "--")
        L = [r"\begin{table}[t]", r"\centering",
             r"\small\setlength{\tabcolsep}{4pt}",
             r"\caption{" + (note + " " if note else "")
             + r"What the examinees scored, under two ways of marking. "
             r"\textsf{prefix\%} gives a term when the examinee produced it in "
             r"order before its first mistake, so an attempt scores a fraction "
             r"of the $k$ terms asked for. \textsf{full} scores the same "
             r"attempt all or nothing: every term right, or none of the credit. "
             r"Within each, \textsf{worst} and \textsf{best} take the least and "
             r"the most of the three attempts and \textsf{mean} averages them, "
             r"so the gap between worst and best is how much of a score is "
             r"luck. \textsf{agree} is not about the score: it says how far the "
             r"three attempts SAID THE SAME THING, right or wrong, since an "
             r"examinee that gives one wrong answer three times over is "
             r"deterministic and one that gives three different wrong answers "
             r"is guessing. Under \textsf{prefix\%} it is how many leading "
             r"terms all three share, as a fraction of $k$; under \textsf{full} "
             r"it is whether the three are identical throughout. An attempt "
             r"with no reply is left out of the agreement, since three refusals "
             r"agree perfectly and that is not the sense meant."
             + "}",
             r"\begin{tabular}{lrrrrrrrr}", r"\toprule",
             r" & \multicolumn{4}{c}{\textsf{prefix\%}} & "
             r"\multicolumn{4}{c}{\textsf{full}} \\",
             r"\cmidrule(lr){2-5}\cmidrule(lr){6-9}",
             r"examinee & worst & mean & best & agree & worst & mean & best & "
             r"agree \\", r"\midrule"]
        for r in out:
            L.append(f"{r['short']} & {p(r['worst'])} & {p(r['mean'])} & "
                     f"{p(r['best'])} & {p(r['pagree'])} & {p(r['fworst'])} & "
                     f"{p(r['fmean'])} & {p(r['fbest'])} & {p(r['fagree'])}"
                     r" \\")
        L += [r"\bottomrule", r"\end{tabular}", r"\end{table}"]
        open(path, "w").write("\n".join(L) + "\n")
        print(f"\nwrote {path}")
    return out
def corr_tex(tab, path):
    """The same table as LaTeX, so a number in the paper is not retyped."""
    # The row labels are LaTeX already, so they are NOT escaped: escaping them
    # turned $K_{\text{cond}}$ into $K\_{\text{cond}}$. Only the free text is.
    esc = lambda s: (s.replace("_", r"\_").replace("%", r"\%")
                     .replace("&", r"\&"))
    raw = lambda s: s
    # \small and a tight column gap: with nine examinees and a row label as
    # long as K(x_{n+1:n+k} | x_{1:n}) + log tau, the default size runs off the
    # page and the last column is lost.
    L = [r"\begin{table}[t]", r"\centering",
         # The size follows the number of columns. Nine examinees fit at
         # \small; thirteen plus the ensemble do not, and the last column ran
         # off the right edge. The row labels are long, so the column gap goes
         # first and the size after.
         (r"\scriptsize\setlength{\tabcolsep}{2.0pt}"
          if len(tab["names"]) > 11 else
          r"\small\setlength{\tabcolsep}{3.4pt}"),
         # WHICH MARKING. A correlation against prefix% and one against full
         # are different numbers, and a table that does not say which cannot be
         # read beside the other.
         r"\caption{" + f"$N = {tab['n_items']}$: " + raw(tab["title"])
         + (r", ranked against \textsf{full} (mean@3), each attempt scored "
            r"all or nothing"
            if tab["kind"] == "f_mean" else
            r", ranked against \textsf{prefix\%} (mean@3), the fraction of "
            r"terms right before the first mistake")
         + (f", as few as {tab['n_min']} where a measure is missing"
            if tab["n_min"] != tab["n_items"] else "")
         # The clause above already names the marking, and this printed the
         # kind as stored, so "f_mean" reached the caption with an unescaped
         # underscore and LaTeX read it as a subscript.
         + ". Spearman rank correlation. "
         + (raw(tab["note"]) if tab["note"] else "") + "}",
         r"\begin{tabular}{l" + "r" * len(tab["names"]) + "}", r"\toprule",
         "measure & "
         # The kind is stored as it is indexed, and "f_mean" carries an
         # underscore LaTeX reads as a subscript. The column is the ensemble
         # either way, and the caption already says which marking was used.
         + " & ".join(r"\textsf{all}" if e == ENSEMBLE else short(e)
                      for e in tab["names"]) + r" \\",
         r"\midrule"]
    for label, needs, cells in tab["rows"]:
        L.append(f"{raw(label)} & "
                 + " & ".join(f"${s:+.2f}$" if s is not None else "--"
                              for s, _ in cells) + r" \\")
    L += [r"\bottomrule", r"\end{tabular}", r"\end{table}"]
    open(path, "w").write("\n".join(L) + "\n")
    print(f"wrote {path}")
    return "\n".join(L)


In [ ]:

# THE ORDER IS THE ORDER THE PIPELINE PRODUCES THEM, so a reader goes down the
# column the way the measurements were made: the dials as set, the lengths of
# what the composer wrote, the dials as typed, then what pricing gives, then
# what annotation gives.
#
# The bar that marks a conditional quantity is dropped wherever the arguments
# are written out, since "given x_{1:n}" is already there and saying it twice
# is noise. It is kept only in the abbreviated forms.
#
# \smash[t] on every hatted symbol: a \widehat adds height to its row, and a
# table where some rows are taller than others for that reason alone looks
# broken.
ROWS_ALL = [
 # The dials as they were SET, separately and together. The sum alone hides
 # which of the two is doing the work: on this battery the transformation dial
 # predicts difficulty and the retrieval dial does not.
 (r"$r^0$", "nothing", lambda b: b.get("want_retrieval")),
 (r"$t^0$", "nothing", lambda b: b.get("want_transformation")),
 (r"$r^0 + t^0$", "nothing", lambda b:
  (b.get("want_retrieval") or 0) + (b.get("want_transformation") or 0)),
 (r"$\ell_{\mathrm{char}}(p^0)$", "generation",
  lambda b: len(b.get("seed") or "") or None),
 (r"$\ell_{\mathrm{char}}(p^{\triangledown})$", "the equivalents search",
  lambda b: len(b["bare"]) if b.get("bare") else None),
 (r"$\ell_{\mathrm{char}}(p^{\blacktriangledown})$", "unquestionability",
  lambda b: b.get("p_star_len")),
 (rf"$\smash[t]{{\hat{{r}}}}_{{{MACHINE_SUB}}}$",
  r"then $r,t$-measuring", lambda b: b.get("r_hat")),
 (rf"$\smash[t]{{\hat{{t}}}}_{{{MACHINE_SUB}}}$", r"unquestionability, then typing", lambda b: b.get("t_hat")),
 (rf"$\smash[t]{{\hat{{r}}_{{{MACHINE_SUB}}}+\hat{{t}}_{{{MACHINE_SUB}}}}}$",
  r"then $r,t$-measuring",
  lambda b: b.get("load")),
 # what pricing gives. K is verified by the EXECUTOR and priced by the
 # reference machine, so it carries the executor: a different claim from the
 # ones below, which count only what the reference machine could run.
 (r"$L_u$", "unquestionability", lambda b: b.get("n_M")),
 # n, not L_u. They are the same length and every other label in the
        # table calls it n, so two names for one thing is one too many.
        (r"$\mathrm{c}(x_{1:n})$", "annotation", lambda b: b.get("c_xpref")),
 (rf"$\tau_{{{MACHINE_SUB}}}$", "annotation", lambda b: b.get("tau")),
# BOTH the composer that wrote the description and the machine that
        # priced it. Written K_V alone it reads as a cost under the verifier,
        # which is not what it is: the description comes from V and the bits
        # are M's.
        (r"$\smash[t]{\hat{K}^{\V}_{\M}}(x_{1:n+k})$", "unquestionability",
         lambda b: b.get("K")),
]

# Table 3 and 4 carry every row of Table 2 as well, so the annotation measures
# can be read against the ones that cost nothing on the same items.
ROWS_H = ROWS_ALL + [
 # what annotation gives, under the reference machine. tau is conditional too:
 # it is the work spent on the program that certified GIVEN the prefix.
 (rf"$\smash[t]{{\hat{{K}}_{{{MACHINE_SUB}}}}}(x_{{n+1:n+k}} \mid x_{{1:n}})$", "annotation",
  lambda b: b.get("K_cond")),
  # the two-term forms. A description is worth what it costs to state AND what
 # it costs to run, and the log keeps the two on one scale, since tau spans 96
 # to 6000 tokens.
 (rf"$\smash[t]{{\widehat{{Kc}}_{{{MACHINE_SUB}}}}}(x_{{n+1:n+k}} \mid x_{{1:n}})$", "annotation",
  lambda b: b.get("h")),
 (rf"$\smash[t]{{\widehat{{Kt}}_{{{MACHINE_SUB}}}}}(x_{{n+1:n+k}} \mid x_{{1:n}})$", "annotation",
  lambda b: (b["K_cond"] + math.log2(b["tau"]))
            if b.get("K_cond") is not None and b.get("tau") else None),
]

CAPTION_ALL = (
 f"Here $\\M$ is {MACHINE_SHOWN}, the reference machine, and $\\V$ is the "
 f"executor that verified the equivalents. "
 r"A quantity written with its arguments does not repeat the bar that marks "
 r"it conditional: $K_\M(x_{n+1:n+k} \mid x_{1:n})$ already says the prefix "
 r"was given, and $K^{|}_\M$ is the same thing abbreviated. "
 rf"$r^0,t^0$ is the cell asked for and $\hat{{r}}_{{{MACHINE_SUB}}},\hat{{t}}_{{{MACHINE_SUB}}}$ the cell measured by "
 r"typing $p^{\blacktriangledown}$; $\ell_{\mathrm{char}}$ is a length in "
 r"characters; $p^0$ is the seed, $p^{\triangledown}$ the shortest verified "
 r"equivalent, $p^{\blacktriangledown}$ the candidate that achieves $K$, which "
 r"is the seed rather than an equivalent on 14 of 62 items. $K(x_{1:n+k})$ is "
 r"$\mathrm{c}(p^{\blacktriangledown})$, so that row and "
 r"$\ell_{\mathrm{char}}(p^{\blacktriangledown})$ are one program measured in "
 r"bits and in characters. $L_u$ is the unquestionability length. The "
 r"reference machine is Qwen3-8B in 4-bit. Examinees: g3.8F is "
 r"gemini-3.8-flash and so on for the Gemini family, haiku is "
 r"claude-haiku-4-5, sonnet is claude-sonnet-5, GPT5nano is gpt-5-nano, olmo2 "
 r"is OLMo-2-1124-7B-Instruct, qwen1.5B is Qwen2.5-1.5B-Instruct. Columns run "
 r"weakest first, by measured accuracy, and \textsf{all} is the mean over the "
 r"three runs of every examinee, per item.")

CAPTION_H = (
 r"Rows as in the previous table, plus "
 r"the ones the annotation measures: $h = K(x_{n+1:n+k} \mid x_{1:n}) + "
 r"\mathrm{c}(x_{1:n})$, its two parts, and $\tau_\M$, the tokens the "
 r"reference machine spent on the program that certified the item. The last "
 r"row adds the log of that work to the cost of the description, on the "
 r"grounds that a description is worth what it costs to state and what it "
 r"costs to run. Throughout, $n$ is $\Lu$, the shortest prefix at which no "
 r"rival survives, so it differs by item and by machine. This table and the "
 r"previous one converge as the annotation finishes.")

CAPTION_NOLAM = (
 r"Certified under THE MACHINE means some program reproduced the "
 r"continuation when the reference machine ran it, having been given the "
 r"prefix. Nothing is compared: a program that runs certifies the item, "
 r"whatever it costs. On the rest nothing the machine was given reproduced "
 r"the continuation, so their $Kc_\M$ is the cost of the listing, which is "
 r"a bound and not a measurement. That matters most for $\tau_\M$: the "
 r"listing is cheap to execute and stands in on the items no rule could "
 r"carry, so pooling the two makes the work look unrelated to difficulty "
 r"when within a band it is strongly related. Rows and columns as in the "
 r"previous tables.")

if ROWS:
    _nc = len([i for i in ITEMS if ITEMS[i].get("n_M") is not None])
    scores_table(ROWS, f"{FIGDIR}/tab_scores_{MACHINE_TAG}.tex",
                 note=f"$N = {_nc}$: the valid items, unquestionable under "
                      f"{MACHINE_SHOWN}.")
    _c = [i for i in ITEMS if ITEMS[i].get("band") not in (None, "p_lam")]
    if _c:
        print()
        scores_table(ROWS, f"{FIGDIR}/tab_scores_comp_{MACHINE_TAG}.tex", only=_c,
                     note=f"$N = {len(_c)}$: the valid items, unquestionable "
                          f"under {MACHINE_SHOWN} and found to be certified "
                          f"under it, so $Kc_\M$ is the cost of a rule rather than "
                          f"of a copy.")
    # No separate table over ROWS_ALL. It held the measures that cost nothing
    # and was split from the annotation table when only some items had an h.
    # Every certified item has one now, so the two tables were over the same
    # thirty items and ROWS_H already contains every row of ROWS_ALL.
    if _hh:
        for _kind in ("worst", "mean", "best", "f_mean"):
            t = corr_table(ROWS, ITEMS, ROWS_H, f"the valid items unquestionable under {MACHINE_SHOWN}, with a $Kc_\M$",
                           CAPTION_H, only=_hh, kind=_kind)
            show_corr(t); corr_tex(t, f"{FIGDIR}/tab_h_{_kind}_{MACHINE_TAG}.tex")

            # The same rows, restricted to items a PROGRAM certified. tau is
        # measured on whichever program won, and the listing wins on the items
        # no rule could carry while costing almost nothing to run, so pooling
        # the two bands makes tau look unrelated to difficulty when within a
        # band it is strongly related.
        _nolam = [i for i in _hh if ITEMS[i].get("band") != "p_lam"]
        if len(_nolam) > 5:
            for _kind in ("worst", "mean", "best", "f_mean"):
                t = corr_table(ROWS, ITEMS, ROWS_H,
                               f"the valid items unquestionable under {MACHINE_SHOWN} and found to be certified under it",
                               CAPTION_NOLAM.replace("THE MACHINE", MACHINE_SHOWN),
                               only=_nolam, kind=_kind)
                show_corr(t); corr_tex(t, f"{FIGDIR}/tab_nolam_{_kind}_{MACHINE_TAG}.tex")
else:
    print("no results file, so no correlations")


### $h$ under two reference machines

One row per item, the two $h$ values joined by an arrow.

An arrow pointing left means the second machine found the item cheaper, which
is what a stronger machine should do.
Whether that leaves $h$ a better
predictor is the question: if the cloud tightens around a downward trend the
measure has improved, and if it flattens it has not.

Red marks a change of band.
A move into $p_\epsilon$ means the second machine
continues the item with no program at all, so $h$ collapses to the cost of the
prefix.

In [ ]:
def shift_plot(items, metrics, results, m_from, m_to, name_from="", name_to="",
               path=None, label_ids=True, field="h",
               field_label=r"$Kc_{\\mathcal{M}}$", y_from=None,
               bare=False, solid=False, drop_listing=False,
               also_pdf=False):
    """How h moves when the reference machine changes, at fixed accuracy.

    One row per item. The x positions are h under the two machines and the y
    position is the fraction of attempts every examinee got right, which does
    NOT depend on the machine: the examinees answered the item, not the
    machine's description of it. So every arrow is horizontal, and the picture
    is of the measure moving under a fixed truth.

    An arrow pointing LEFT means the second machine describes the item at less
    cost, which is what a stronger machine should do. Whether that leaves h a better
    predictor is the point of the figure: if the cloud tightens around a
    downward trend the measure has improved, and if it flattens it has not.
    """
    import matplotlib.pyplot as plt
    import numpy as np
    import collections, statistics as st

    acc = collections.defaultdict(list)
    for r in results:
        acc[r["item"]].append(ACC(r))

    if y_from is None:
        raise ValueError("shift_plot needs y_from: without it the labels go to "
                         "m_to whichever machine is reporting")
    _label_at_to = (str(y_from) == str(name_to or m_to))

    rows = []
    for i in items:
        a = (metrics.get(i) or {}).get(m_from) or {}
        b = (metrics.get(i) or {}).get(m_to) or {}
        va = field(a) if callable(field) else a.get(field)
        vb = field(b) if callable(field) else b.get(field)
        if drop_listing:
            # a listing is not a description, so it is not a point on an axis
            # that reads as the cost of one
            if a.get("band") in (None, "p_lam"): va = None
            if b.get("band") in (None, "p_lam"): vb = None
        if (va is None and vb is None) or i not in acc:
            continue
        rows.append((va, vb, st.mean(acc[i]), i,
                     a.get("band"), b.get("band")))
    if not rows:
        print(f"no item has {field} under both machines yet")
        return None

    # LABELS THAT DO NOT SIT ON EACH OTHER. Several items share an accuracy,
    # so their rows share a y and their names landed on top of one another.
    # Each row gets a rank among the rows at its y, and the label is lifted by
    # that rank with a leader line back to the point.
    _aty = {}
    for _r in rows:
        _x = _r[1] if _r[1] is not None else _r[0]
        _aty.setdefault(round(_r[2], 6), []).append((_x, _r[3]))
    # RANKED BY X WITHIN A ROW, and the stack wraps. Sorting by id put
    # neighbours at the same height and the eye could not tell which label
    # belonged to which point; ordering by position along the axis puts them
    # in reading order. The wrap keeps a row of eight from running off the
    # top of the axes.
    _rank = {}
    _STACK = 4
    for _y, _rs in _aty.items():
        for _n, (_x, _id) in enumerate(sorted(_rs)):
            _rank[_id] = _n % _STACK

    DOWN, UP = "#1a7f37", "#b3242b"
    C_FROM, C_TO = "#7a5195", "#ef8a3d"
    # the arrow says which WAY the measure moved, which is the whole question.
    # A change of band is not marked: it follows from the move rather than
    # adding to it, and two things encoded on one arrow read as neither.
    fig, ax = plt.subplots(figsize=(10.4, 5.2))
    for h0, h1, y, i, b0, b1 in rows:
        lam0, lam1 = (b0 == "p_lam"), (b1 == "p_lam")
        # An item is CERTIFIED under a machine when a program it could run
        # cost less than writing the continuation out. Where it is not, the
        # value is the cost of a copy and not of a rule, which is a different
        # quantity on the same axis. Those are drawn faint: kept, because
        # dropping them would hide the disagreement, but not competing for the
        # eye with the items the figure is about.
        # A LISTING IS NOT A MEASUREMENT, so anything touching one recedes:
        # the arrow and the square alike. A circle is always at full strength,
        # even the one at the other end of a dashed arrow, because that end IS
        # certified. The eye should land on the certified values.
        _dim = 1.0 if solid else (0.25 if (lam0 or lam1) else 1.0)
        if h0 is None or h1 is None:
            # only one machine has it: a point, no arrow, and a cross through
            # it so it cannot be read as a pair that happened not to move
            hx, cx_, bx = ((h1, C_TO, lam1) if h0 is None else (h0, C_FROM, lam0))
            _d1 = 0.22 if bx else 1.0
            _a1 = 1.0 if solid else (0.25 if bx else 1.0)
            ax.scatter([hx], [y], s=46 if bx else 42, alpha=_a1,
                       marker="s" if bx else "o",
                       facecolor="none", edgecolor=cx_, lw=1.4, zorder=5)
            # the cross said "only one machine has a value here", which a lone
            # point already says once the listings are not drawn
            if not drop_listing:
                ax.scatter([hx], [y], s=16, marker="x", color=cx_, lw=1.1,
                           alpha=_a1, zorder=6)
            if label_ids:
                # only one point exists here, so the side is free; keep it left
                _k = _rank.get(i, 0)
                ax.annotate(i.replace("v17-", ""), (hx, y),
                            textcoords="offset points", xytext=(0, 9 + 9 * _k),
                            ha="center", va="bottom", fontsize=6.5, color="0.45",
                            arrowprops=dict(arrowstyle="-", lw=.5, color="0.75",
                                            shrinkA=0, shrinkB=3)
                            if _k else None)
            continue
        ax.annotate("", xy=(h1, y), xytext=(h0, y),
                    arrowprops=dict(arrowstyle="-|>", lw=1.3,
                                    linestyle="--" if (lam0 or lam1) else "-",
                                    color=DOWN if h1 < h0 else UP,
                                    alpha=_dim * .9,
                                    shrinkA=4, shrinkB=4))
        # THE MARKERS ARE ALWAYS AT FULL STRENGTH. Only the arrow is faint
        # when one end is a listing: what is uncertain there is the COMPARISON,
        # since the ends are different kinds of quantity, not either endpoint.
        # Dimming the markers as well made the items hard to find on the axis.
        ax.scatter([h0], [y], s=46 if lam0 else 42,
                   alpha=1.0 if solid else (0.25 if lam0 else 1.0),
                   marker="s" if lam0 else "o",
                   facecolor="none", edgecolor=C_FROM, lw=1.4, zorder=4)
        ax.scatter([h1], [y], s=46 if lam1 else 42,
                   alpha=1.0 if solid else (0.25 if lam1 else 1.0),
                   marker="s" if lam1 else "o",
                   facecolor="none", edgecolor=C_TO, lw=1.4, zorder=5)
        if label_ids:
            # THE LABEL SITS AT THE REPORTING MACHINE'S POINT, whichever side
            # that falls on. It was pinned to min(h0, h1), the left end, so on
            # a figure about the 14B half the labels named the 8B's value and
            # the eye had to cross the arrow to find the one meant. Which end
            # is which is `y_from`: the x axis always runs 8B to 14B, so the
            # 14B is h1 and the 8B is h0.
            _at = h1 if _label_at_to else h0
            _k = _rank.get(i, 0)
            _dy = 9 + 9 * _k
            _col = ("0.45" if solid else
                    ("0.72" if (lam0 and lam1) else "0.45"))
            ax.annotate(i.replace("v17-", ""), (_at, y),
                        textcoords="offset points", xytext=(0, _dy),
                        ha="center", va="bottom", fontsize=6.5, color=_col,
                        arrowprops=dict(arrowstyle="-", lw=.5, color="0.75",
                                        shrinkA=0, shrinkB=3)
                        if _k else None)

    ax.scatter([], [], s=42, marker="o", facecolor="none", edgecolor=C_FROM,
               lw=1.4, label=f"under {name_from or m_from}")
    ax.scatter([], [], s=42, marker="o", facecolor="none", edgecolor=C_TO,
               lw=1.4, label=f"under {name_to or m_to}")
    # the listing entries only when a listing is drawn
    if not drop_listing:
        ax.scatter([], [], s=46, marker="s", facecolor="none", edgecolor="0.35",
                   lw=1.4, label="square: the listing $p^{\\lambda}$, no program ran")
    ax.plot([], [], color=DOWN, lw=1.3,
            label=f"less cost under {name_to or m_to}")
    ax.plot([], [], color=UP, lw=1.3,
            label=f"more cost under {name_to or m_to}")
    if not drop_listing:
        ax.plot([], [], color="0.4", lw=1.3, linestyle="--",
                label="dashed: one end is a listing, so the ends differ in kind")
    if any(h0 is None or h1 is None for h0, h1, *_ in rows) and not drop_listing:
        ax.scatter([], [], s=42, marker="x", color="0.35", lw=1.1,
                   label="cross: only one machine certified this item")

    ax.set_xlabel(f"{field_label}, bits")
    ax.set_ylabel("% correct, over all models and attempts")
    ax.set_ylim(-0.04, 1.04)
    ax.grid(alpha=.25)
    # INSIDE once the listing entries are gone: four entries fit in the empty
    # upper right, and a figure going into a paper should not carry a margin
    # of legend the template has to crop.
    if drop_listing:
        ax.legend(fontsize=7.5, loc="upper right", framealpha=.92,
                  borderaxespad=0.5)
    else:
        ax.legend(fontsize=7.5, loc="upper left", bbox_to_anchor=(1.01, 1.0),
                  borderaxespad=0, frameon=False)
    _dx = [h1 - h0 for h0, h1, *_ in rows if h0 is not None and h1 is not None]
    _solo = sum(1 for h0, h1, *_ in rows if h0 is None or h1 is None)
    if not bare:
        ax.set_title(
            f"How {field_label} moves from {name_from or m_from} to "
            f"{name_to or m_to}, {len(rows)} items\n"
            f"accuracy is a property of the item and does not move, so every "
            f"arrow is horizontal"
            + (f"   (answers as scored for {y_from})" if y_from else ""),
            fontsize=9.5)
        fig.text(0.5, -0.01,
                 f"median move {st.median(_dx):+.0f} bits, "
                 f"{sum(1 for d in _dx if d < 0)} at less cost and "
                 f"{sum(1 for d in _dx if d > 0)} at more under "
                 f"{name_to or m_to}. An arrow pointing left means the second "
                 f"machine describes the item at less cost."
                 + (f" {_solo} item(s) only one machine certified, drawn as a "
                    f"crossed point." if _solo else ""),
                 ha="center", va="top", fontsize=8, color="0.3")
    fig.tight_layout(rect=(0, 0, 1, 1) if drop_listing else (0, 0, 0.80, 1))
    if path:
        fig.savefig(path, dpi=150, bbox_inches="tight")
        print(f"wrote {path}")
        if also_pdf:
            # VECTOR, AND CROPPED TO THE INK. A figure going into a paper is
            # scaled there, so a raster loses at the scale it is printed, and
            # the white margin fights whatever spacing the template sets.
            _pdf = path.rsplit(".", 1)[0] + ".pdf"
            fig.savefig(_pdf, bbox_inches="tight", pad_inches=0.01)
            print(f"wrote {_pdf}")
    return fig

WINDOW_LO = 1.0 / 3.0    # below this the examinee is hopeless
WINDOW_HI = 2.0 / 3.0    # above it, reliable
# THE COLUMNS OF THE WINDOWS TABLE, in order: (machine id, tag, shown).
# Named rather than discovered, so that adding a machine to the metrics file
# does not silently widen a table that has to fit a page.
WINDOW_MACHINES = [
    ("qwen3-8b-4bit-think",  "qwen3-8b",        "Qwen3-8B"),
    ("qwen3-14b-4bit-think", "qwen3-14b",       "Qwen3-14B"),
    ("phi-4-reasoning",      "phi-4-reasoning", "Phi-4"),
]

WINDOW_MIN_ITEMS = 10    # fewer than this and a two-parameter fit is not
                         # worth reporting

def windows_for(mid, tag, kind="full"):
    """Both edges per examinee, from the curve fitted on that machine's
    certified items. `kind` is "full", all k_f terms right or nothing, which is
    what a window in these terms means; "prefix" gives partial credit and moves
    both edges outward."""
    # IMPORTED HERE. bins_plot imports it inside itself, so at module scope
    # the name does not exist, and the bare `except Exception: continue` around
    # the fit swallowed the NameError once per examinee and returned an empty
    # table with no complaint.
    from scipy.optimize import curve_fit

    kt, order = {}, None
    # BAT is a LIST here, and the machine's own fields are read off each item
    _byid = {b["id"]: b for b in BAT}
    for i, b in _byid.items():
        if not _is_valid(b):
            continue
        S = (MET.get(i) or {}).get(mid) or {}
        if S.get("band") in (None, "p_lam"):
            continue
        tau = _cert_of(S).get("tau")
        if S.get("K_cond") is None or not tau:
            continue
        kt[i] = S["K_cond"] + math.log2(tau)
    if not kt:
        return {}, []
    order = sorted(kt, key=lambda i: kt[i])

    # the prefix this machine needs, as everywhere else
    need = {i: (_byid[i].get(f"n_M_{tag}") or 0) for i in kt}
    best = {}
    _rows = (_res or {}).get("rows", [])
    for r in _rows:
        if r["item"] not in kt:
            continue
        n = r.get("n_M")
        if n is None or n < need[r["item"]]:
            continue
        k = (r["examinee"], r["item"])
        if best.get(k) is None or n < best[k]:
            best[k] = n
    acc = collections.defaultdict(lambda: collections.defaultdict(list))
    for r in _rows:
        if r["item"] in kt and r.get("n_M") == best.get((r["examinee"], r["item"])):
            frac = r["n_correct"] / r["k"]
            acc[r["examinee"]][r["item"]].append(
                (1.0 if frac >= 0.999 else 0.0) if kind == "full" else frac)

    # THE SAME LOGISTIC IN BITS as agent_curve, with the same prior at (0,1),
    # so a window and an ability come off one shape and not two.
    def curve(x, th, s):
        return 1.0 / (1.0 + np.exp(np.clip((np.asarray(x) - th) / s, -50, 50)))

    def crossing(b, k, y):
        """where the curve falls through y, or None if it never does"""
        g = np.linspace(0.01, kt[order[-1]] * 1.6, 200000)
        below = np.where(curve(g, b, k) <= y)[0]
        return float(g[below[0]]) if len(below) else None

    out = {}
    for e, d in acc.items():
        mine = [i for i in order if i in d]
        if len(mine) < WINDOW_MIN_ITEMS:
            continue
        xs = np.array([kt[i] for i in mine])
        ys = np.array([st.mean(d[i]) for i in mine])
        try:
            _n = 2 * len(xs)
            _X = np.concatenate([xs, np.zeros(_n)])
            _Y = np.concatenate([ys, np.ones(_n)])
            p, _ = curve_fit(curve, _X, _Y,
                             p0=[max(np.median(xs), 1.0), 30.0],
                             bounds=([0.1, 1.0], [1e4, 1e4]), maxfev=40000)
        except Exception as _e:
            print(f"    {e}: the curve would not fit, left out ({_e})")
            continue
        hi = crossing(*p, WINDOW_HI)       # end of the infra window
        lo = crossing(*p, WINDOW_LO)       # start of the supra window
        out[e] = dict(infra_hi=hi, supra_lo=lo,
                      span=(lo - hi) if (lo and hi) else None,
                      floor=kt[mine[0]], ceiling=kt[mine[-1]],
                      mean=float(ys.mean()), n=len(mine), of=len(order))
    return out, order

def windows_table(path=None, kind="full"):
    """One table, a column group per machine in WINDOW_MACHINES, with the span
    between the two windows in the middle of each group.

    An examinee is listed only where it answered at least WINDOW_MIN_ITEMS of
    that machine's certified set, so a dash is missing coverage and not a
    missing window.
    """
    cols = []
    for mid, tg, nm in WINDOW_MACHINES:
        W, order = windows_for(mid, tg, kind)
        if W:
            cols.append((nm, W))
    if not cols:
        print("  no certified items under any machine yet"); return
    names = sorted({e for _, W in cols for e in W},
                   key=lambda e: -max((W[e]["mean"] for _, W in cols
                                       if e in W), default=0))
    nc = len(cols)
    spec = "@{}l " + " ".join([r"r@{\,--\,}l c r@{\,--\,}l"] * nc) + "@{}"
    head1 = " & " + " & ".join(r"\multicolumn{5}{c}{%s}" % nm for nm, _ in cols)
    rules = "".join(r"\cmidrule(lr){%d-%d}" % (2 + 5 * k, 6 + 5 * k)
                    for k in range(nc))
    head2 = (" & " + " & ".join([r"\multicolumn{2}{c}{infra} & span "
                                 r"& \multicolumn{2}{c}{supra}"] * nc))
    L = [r"\begin{table}[t]\centering\small",
         r"\caption{Where each examinee is reliable and where it is hopeless, in "
         r"bits on the scale of the items. The INFRA window runs from the cheapest "
         r"item to the $\widehat{Kt}_\M$ at which the fitted response curve falls "
         r"through two thirds; the SUPRA window runs from where it falls through "
         r"one third to the dearest item there is, and the span between them is "
         r"the range over which the examinee is neither. Both are read off the "
         r"curve rather than off a run of item scores, because a run breaks on "
         r"one lucky item. The curve is a logistic in bits, $1/(1+e^{(x-\theta)/s})$, "
         r"fitted with as many pseudo-observations at $(0,1)$ as there are items: "
         r"an item costing nothing is answered by anyone, and stating that as a "
         r"prior keeps the curve near 1 at the origin without tying its width to "
         r"its location. Both edges belong to the reference machine, since "
         r"$\widehat{Kt}_\M$ does and the machines certify different items, so "
         r"the groups are separate quantities and not readings of one. An edge "
         r"GREYED is one the curve reaches only beyond the dearest item the "
         r"battery holds, so it describes no item that exists; a dash is a curve "
         r"that never crosses that level at all. An attempt counts only if every "
         r"one of the $\kf$ terms is right.}",
         r"\label{tab:windows}",
         r"\begin{tabular}{%s}\toprule" % spec,
         head1 + r" \\", rules, head2 + r" \\",
         r"\addlinespace[2pt]", r"\midrule"]

    def _v(x, grey=False):
        if x is None: return r"---"
        t = f"{x:.0f}"
        return (r"\textcolor{gray}{" + t + "}") if grey else t

    for e in names:
        cells = []
        for _, W in cols:
            w = W.get(e)
            if not w:
                cells += [r"---"] * 5; continue
            hi_out = w["infra_hi"] is not None and w["infra_hi"] > w["ceiling"]
            lo_out = w["supra_lo"] is not None and w["supra_lo"] > w["ceiling"]
            cells += [_v(w["floor"]), _v(w["infra_hi"], hi_out),
                      _v(w["span"], hi_out or lo_out),
                      _v(w["supra_lo"], lo_out), _v(w["ceiling"])]
        L.append(f"{short(e)} & " + " & ".join(cells) + r" \\")
    note = "; ".join(
        f"at most {max((w['n'] for w in W.values()), default=0)} of the "
        f"{max((w['of'] for w in W.values()), default=0)} certified under {nm}"
        for nm, W in cols)
    L += [r"\bottomrule", r"\end{tabular}",
          r"\par\vspace{2pt}\footnotesize Fitted on the items each examinee "
          f"was asked: {note}. Where annotation has run ahead of evaluation the "
          "newest certified items carry no answers yet and are not in the fit.",
          r"\end{table}"]
    out = "\n".join(L) + "\n"
    if path:
        open(path, "w").write(out); print(f"  wrote {path}")
    return out

windows_table(f"{FIGDIR}/tab_windows{FIG_SUFFIX}.tex")


In [ ]:
_SHOWN = {"qwen3-8b-4bit-think":  "Qwen3-8B",
          "qwen3-14b-4bit-think": "Qwen3-14B",
          "phi-4-reasoning":      "Phi-4-reasoning"}
SHIFT_PARTNER = {"qwen3-14b-4bit-think": "qwen3-8b-4bit-think",
                 "qwen3-8b-4bit-think":  "phi-4-reasoning",
                 "phi-4-reasoning":      "qwen3-14b-4bit-think"}

_M2 = SHIFT_PARTNER.get(MACHINE_ID, "qwen3-14b-4bit-think")
_M2_SHOWN = _SHOWN.get(_M2, _M2)

# The arrow runs the same way whichever figure an item appears in, so a reader
# comparing two of them is not reading one backwards.
_ORDER = ["qwen3-8b-4bit-think", "phi-4-reasoning", "qwen3-14b-4bit-think"]
_pair = sorted([MACHINE_ID, _M2],
               key=lambda m: _ORDER.index(m) if m in _ORDER else 99)
_M_FROM, _M_TO = _pair[0], _pair[1]

# EITHER machine, not both. Unquestionability is machine-relative, so each
# certifies items the other does not, and showing only the intersection would
# hide the disagreement. An item measured under one is drawn as a crossed point
# with no arrow. It still needs an accuracy to be placed, so one never
# administered does not appear whatever its h.
_both = [i for i in MET
         if ((MET[i].get(MACHINE_ID) or {}).get("h") is not None
             or (MET[i].get(_M2) or {}).get("h") is not None)
         and i in ITEMS]
if _both:
    def _kt(S):
        """Conditional Levin complexity: the description plus the log of the
        work needed to run it. The only one of the three that does not lose
        resolution as the machine gets stronger, because a program that costs
        nothing to state still costs something to execute."""
        if S.get("K_cond") is None: return None
        _t = _tau_of(S)
        return S["K_cond"] + math.log2(_t) if _t else None

    shift_plot(sorted(_both), MET, ROWS, _M_FROM, _M_TO,
               _SHOWN[_M_FROM], _SHOWN[_M_TO], field_label=r"$\hat{h}$",
               y_from=MACHINE_SHOWN,
               path=f"{FIGDIR}/fig_shift_h{FIG_SUFFIX}.png")
    shift_plot(sorted(_both), MET, ROWS, _M_FROM, _M_TO,
               _SHOWN[_M_FROM], _SHOWN[_M_TO], field=_kt,
               field_label=r"$\widehat{Kt}(x_{n+1:n+k} \mid x_{1:n})$",
               y_from=MACHINE_SHOWN,
               path=f"{FIGDIR}/fig_shift_kt{FIG_SUFFIX}.png")
    shift_plot(sorted(_both), MET, ROWS, _M_FROM, _M_TO,
               _SHOWN[_M_FROM], _SHOWN[_M_TO], field="K_cond",
               field_label=r"$\mathrm{c}(p^{\overline{\,\blacktriangledown}}"
                           r" \mid x_{1:n})$",
               y_from=MACHINE_SHOWN,
               path=f"{FIGDIR}/fig_shift_cp{FIG_SUFFIX}.png")

    # FOR THE PAPER. The same figure restricted to items CERTIFIED under at
    # least one machine, so every value drawn is the cost of a description the
    # machine ran rather than of a copy. Nothing is a listing, so nothing
    # recedes and everything is at full strength; an arrow appears where both
    # machines certified, a crossed point where only one did. The title and the
    # line beneath are left off, and written to a .tex beside it, so the
    # caption can carry them.
    _cp = sorted(i for i in _both
                 if any(((MET.get(i) or {}).get(m) or {}).get("band")
                        not in (None, "p_lam") for m in (_M_FROM, _M_TO)))
    if _cp:
        shift_plot(_cp, MET, ROWS, _M_FROM, _M_TO,
                   _SHOWN[_M_FROM], _SHOWN[_M_TO], field=_kt,
                   field_label=r"$\widehat{Kt}_\mathcal{M}"
                               r"(x_{n+1:n+k} \mid x_{1:n})$",
                   bare=True, solid=True, drop_listing=True,
                   also_pdf=True, y_from=MACHINE_SHOWN,
                   path=f"{FIGDIR}/fig_shift_kt_paper{FIG_SUFFIX}.png")
        _b = sum(1 for i in _cp
                 if all(((MET.get(i) or {}).get(m) or {}).get("band")
                        not in (None, "p_lam") for m in (_M_FROM, _M_TO)))
        # ONLY THE ITEMS BOTH MACHINES CERTIFIED. K_cond exists on a listing
        # item too, so taking every pair would average moves between two
        # copies into a figure about descriptions, and would not match the
        # count of arrows the sentence above gives.
        _d = []
        for i in _cp:
            if any(((MET.get(i) or {}).get(m) or {}).get("band")
                   in (None, "p_lam") for m in (_M_FROM, _M_TO)): continue
            a_ = _kt((MET.get(i) or {}).get(_M_FROM) or {})
            b_ = _kt((MET.get(i) or {}).get(_M_TO) or {})
            if a_ is not None and b_ is not None: _d.append(b_ - a_)
        _cap = (
            r"How $\widehat{Kt}$, the cost of the description that certified "
            r"an item plus the log of the work its execution took, moves when "
            r"the reference machine changes. One row an item, at its "
            r"accuracy over every examinee and attempt, which is a property of "
            r"the item and does not move, so every arrow is horizontal. Only "
            r"items certified under at least one machine are drawn, so each "
            r"value is the cost of a description the machine ran and not of a "
            f"copy: {len(_cp)} of them, {_b} certified under both and joined "
            f"by an arrow, {len(_cp) - _b} under one and drawn as a lone "
            r"point. Where a machine did not certify an item there is no "
            r"value to draw, so that end is absent rather than shown as a "
            r"listing. An arrow "
            f"pointing left means {_SHOWN[_M_TO]} describes the item at less "
            f"cost; the median move is {st.median(_d):+.0f} bits over the "
            f"{len(_d)} items both certified, "
            f"{sum(1 for d in _d if d < 0)} at less and "
            f"{sum(1 for d in _d if d > 0)} at more.")
        open(f"{FIGDIR}/fig_shift_kt_paper{FIG_SUFFIX}.tex", "w").write(
            _cap + "\n")
        print(f"  wrote the caption to fig_shift_kt_paper{FIG_SUFFIX}.tex")
else:
    print(f"no item has an h under both {MACHINE_SHOWN} and {_M2_SHOWN} yet")


## 5b · The items themselves

Two long tables, one for the items that certified and one for those that did
not.
The second is the more interesting to read: an item fails certification
for a reason that is usually visible in its sequence, and a table showing only
the survivors hides what the pipeline rejects.

Landscape longtables, written as `.tex` to be `\input{}` into the write-up.
They need `longtable`, `pdflscape`, `amssymb` and a `\M` macro.

In [ ]:
# The composer writes real mathematical symbols, so a rule can carry floor
# brackets, an arrow or a not-equals. pdflatex refuses them outright rather
# than dropping them, so the table fails to build unless they are mapped.
_UNI = {"\u230a": r"$\lfloor$", "\u230b": r"$\rfloor$",
        "\u2265": r"$\geq$", "\u2264": r"$\leq$", "\u2260": r"$\neq$",
        "\u2794": r"$\to$", "\u2192": r"$\to$", "\u21a6": r"$\mapsto$",
        "\u00d7": r"$\times$", "\u2212": "-", "\u2026": r"\ldots",
        "\u2192": r"$\to$", "\u2261": r"$\equiv$", "\u221a": r"$\sqrt{}$",
        "\u00f7": r"$\div$", "\u2032": "'", "\u00b2": r"$^2$",
        "\u00b3": r"$^3$", "\u2080": r"$_0$", "\u2081": r"$_1$",
        "\u2099": r"$_n$", "\u03b1": r"$\alpha$", "\u03b2": r"$\beta$",
        "\u2308": r"$\lceil$", "\u2309": r"$\rceil$",
        "\u2019": "'", "\u2018": "`", "\u201c": "``", "\u201d": "''",
        "\u2013": "--", "\u2014": "---", "\u00b7": r"$\cdot$"}

def _tex(s, n=None):
    """Escape for LaTeX, and cut to n characters on a word boundary."""
    s = " ".join(str(s or "").split())
    if n and len(s) > n:
        s = s[:n].rsplit(" ", 1)[0] + "\u2026"
    for a, b in (("\\", r"\textbackslash "), ("&", r"\&"), ("%", r"\%"),
                 ("$", r"\$"), ("#", r"\#"), ("_", r"\_"), ("{", r"\{"),
                 ("}", r"\}"), ("~", r"\textasciitilde "),
                 ("^", r"\textasciicircum "), ("|", r"\textbar "),
                 ("<", r"\textless "), (">", r"\textgreater ")):
        s = s.replace(a, b)
    for a, b in _UNI.items():
        s = s.replace(a, b)
    # anything left outside Latin-1 would stop pdflatex, so it is replaced by
    # a marker rather than silently dropped: a rule with a symbol missing
    # reads as a different rule.
    s = "".join(c if ord(c) < 256 else r"\textbf{?}" for c in s)
    return s

def why_open(b, rec, delta=22.7):
    """Why this item did not certify, from its own pricing record.

    Read in order, since more than one can be true and the first is the one
    that decided it.
    """
    if rec is None:
        return "no $K$ measured"
    if not b.get("bare"):
        return "no verified rewriting"
    per = rec.get("per_L") or {}
    cert = set(rec.get("certify_at") or [])
    marg = {int(k): v for k, v in (rec.get("margins") or {}).items()
            if v is not None}
    # a rival the composer wrote, not the structural one: two readings of the
    # sequence that the machine finds equally cheap
    for L in sorted(cert):
        for bl in (per.get(str(L)) or {}).get("blockers") or []:
            if bl.get("kind") != "structural":
                return "a composer rival blocks"
    # a prefix where the rule would have won, but no rival was asked there, so
    # no verdict could be given
    if [L for L, m in marg.items() if m > delta and L not in cert]:
        return "no rival tested where it would pass"
    if marg and max(marg.values()) <= delta:
        return "no prefix it has is long enough"
    return "the structural rival is inside the budget"

def item_table(items, ids, caption, label, path=None, terms_chars=90,
               rule_chars=150, why=None):
    """One row per item: what it is, where it sits, what it costs.

    Two of these are wanted, certified and not, because the second is the more
    interesting to read: an item that failed certification failed for a reason
    visible in its sequence, and a table that shows only the survivors hides
    what the pipeline rejects.

    Long, so it is a landscape longtable. The sequence is cut at
    `terms_chars`, which is enough to see the pattern, and the rule at
    `rule_chars`.
    """
    L = [r"\begin{landscape}",
         r"\scriptsize\setlength{\tabcolsep}{2.5pt}"
         r"\renewcommand{\arraystretch}{1.05}",
         (r"\begin{longtable}{@{}p{11mm} p{6mm} p{9mm} p{7mm} p{7mm} p{7mm}"
          r" p{26mm} p{30mm} p{54mm} p{40mm}@{}}" if why else
          r"\begin{longtable}{@{}p{11mm} p{6mm} p{9mm} p{7mm} p{7mm} p{7mm}"
          r" p{30mm} p{62mm} p{52mm}@{}}"),
         r"\caption{" + caption + r"}\label{" + label + r"}\\",
         r"\toprule"]
    head = (rf"\textbf{{id}} & \textbf{{$r^0t^0$}} & \textbf{{$\hat{{r}}_{{{MACHINE_SUB}}}/\hat{{t}}_{{{MACHINE_SUB}}}$}} & "
            r"\textbf{$L_u$} & \textbf{$K_\V$} & \textbf{$Kc_\M$} & "
            + (r"\textbf{why not} & " if why else "")
            + r"\textbf{theme} & \textbf{first terms} & "
            r"\textbf{$p^{\blacktriangledown}$, the description $K$ measures}\\")
    _nc = 10 if why else 9
    L += [head, r"\midrule", r"\endfirsthead",
          r"\multicolumn{" + str(_nc) + r"}{l}{\footnotesize\itshape Table \ref{" + label
          + r"}, continued}\\", r"\toprule", head, r"\midrule", r"\endhead",
          r"\midrule\multicolumn{" + str(_nc) + r"}{r}{\footnotesize\itshape continues}\\",
          r"\endfoot", r"\bottomrule", r"\endlastfoot"]

    for i in ids:
        b = items[i]
        rt = f"r{b.get('want_retrieval')}t{b.get('want_transformation')}"
        hat = ("--" if b.get("r_hat") is None
               else f"{b['r_hat']:.1f}/{b['t_hat']:.1f}")
        nu = "--" if b.get("n_M") is None else str(b["n_M"])
        K = "--" if b.get("K") is None else f"{b['K']:.0f}"
        h = "--" if b.get("h") is None else f"{b['h']:.0f}"
        rule = (b.get("p_star") or b.get("bare") or b.get("seed") or "")
        L.append(" & ".join([
            r"\texttt{" + _tex(i.replace("v17-", "")) + "}",
            rt, hat, nu, K, h]
            + ([_tex((why or {}).get(i, ""), 34)] if why else [])
            + [_tex(b.get("theme"), 34),
            r"\texttt{\scriptsize " + _tex(", ".join(b.get("terms") or [])[:terms_chars + 30], terms_chars) + "}",
            _tex(rule, rule_chars)]) + r"\\")
    L += [r"\end{longtable}", r"\end{landscape}"]
    out = "\n".join(L) + "\n"
    if path:
        open(path, "w").write(out)
        print(f"wrote {path}: {len(ids)} items")
    return out


In [ ]:
if ITEMS_ALL:
    # Four tables, one per stage of the chain, so a reader can see what each
    # filter removed rather than only what survived. The first sentence of each
    # caption says exactly which subset it is.
    _valid_ids = set(ITEMS)                       # VALID_ONLY made ITEMS the valid ones
    _cert  = {i for i in _valid_ids if ITEMS[i].get("n_M") is not None}
    _comp  = {i for i in _cert if ITEMS[i].get("band") not in (None, "p_lam")}
    _inval = [i for i in ITEMS_ALL if i not in _valid_ids]
    _quest = sorted(_valid_ids - _cert)
    _lam   = sorted(_cert - _comp)
    _cmp   = sorted(_comp)

    item_table(ITEMS_ALL, sorted(_inval),
        r"The " + str(len(_inval)) + r" items that are NOT VALID: run cold a "
        r"second time, the seed did not reproduce the sequence the first run "
        r"produced. Validity belongs to the rule and the executor and not to "
        r"any reference machine, so these are excluded before any machine is "
        r"involved and do not appear in any other table. They are listed "
        r"because a rule that cannot reproduce its own sequence is worth "
        r"seeing.",
        "tab:items-invalid", path=f"{FIGDIR}/tab_items_invalid_{MACHINE_TAG}.tex")

    item_table(ITEMS, _quest,
        r"The " + str(len(_quest)) + r" items that are VALID but QUESTIONABLE "
        r"under " + MACHINE_SHOWN + r". A rival inside the budget reproduces "
        r"the prefix and then diverges at every prefix tested, so the intended "
        r"continuation is not the only one this machine finds cheap and an "
        r"examinee answering differently may not be wrong. Questionability is "
        r"relative to the machine: a different $\M$ gives a different set.",
        "tab:items-questionable", path=f"{FIGDIR}/tab_items_questionable_{MACHINE_TAG}.tex",
        why={i: why_open(ITEMS[i], PRI.get(i)) for i in _quest})

    item_table(ITEMS, _lam,
        r"The " + str(len(_lam)) + r" items that are VALID and UNQUESTIONABLE "
        r"under " + MACHINE_SHOWN + r" but NOT CERTIFIED under it. No "
        r"program the machine was given reproduced the continuation when it "
        r"was run, so their $Kc_\M$ is the cost of $p^\lambda$, the "
        r"listing, which is a bound and not a measurement.",
        "tab:items-listing", path=f"{FIGDIR}/tab_items_listing_{MACHINE_TAG}.tex")

    item_table(ITEMS, _cmp,
        r"The " + str(len(_cmp)) + r" items that are VALID, UNQUESTIONABLE and "
        r"CERTIFIED under " + MACHINE_SHOWN + r". A program the machine "
        r"could execute cost less than writing the continuation out, so $Kc_\M$ is "
        r"the cost of a rule. These are the items every correlation in this "
        r"document rests on.",
        "tab:items-certified", path=f"{FIGDIR}/tab_items_certified_{MACHINE_TAG}.tex")


## 5c · What never got answered

Every non-answer is scored as wrong, which is what the correlations above are
computed on.
That is the conservative reading.
A refusal or a truncation is not
evidence that the model could not do the item, so counting it as a failure can
only weaken a correlation between a difficulty measure and accuracy, never
manufacture one.

It is recorded so a reader can see how much of a score is ability and how much
is the model never having answered.
The last column gives the score on the rows
that did come back, which is what the examinee would have scored had nothing
gone wrong.

In [ ]:
def short(name):
    """A short column label. The caption says what each one is.

      gemini-3.5-flash-lite        ->  g3.5FL
      claude-haiku-4-5-20251001    ->  haiku
      claude-sonnet-5              ->  sonnet
      gpt-5-nano                   ->  gpt5na
      allenai/OLMo-2-1124-7B       ->  olmo2
      Qwen/Qwen2.5-1.5B-Instruct   ->  q1.5B
      claude-opus-5-fallback       ->  opus*

    Short enough that a table of nine examinees and a `needs' column does not
    run off the page, which is what a longer scheme did.
    """
    if name == ENSEMBLE:
        return "all"
    # "-fallback" is the same model with server-side fallbacks asked for. It
    # has to be told apart from the plain one or a table shows two rows called
    # "opus" and a reader cannot say which is which.
    if name.endswith("-fallback"):
        return short(name[:-len("-fallback")]) + "*"
    n = name.split("/")[-1]
    m = re.match(r"gemini-([0-9.]+)-(.+)", n)
    if m:
        return "g" + m.group(1) + "".join(w[0].upper()
                                          for w in m.group(2).split("-"))
    m = re.match(r"claude-([a-z]+)", n, re.I)
    if m:
        return m.group(1).lower()
    m = re.match(r"gpt-([0-9]+)[.0-9]*-([a-z]+)", n, re.I)
    if m:
        # `sol' is a name and survives whole; everything else is cut to two,
        # which is enough to tell nano from mini.
        _suf = m.group(2).lower()
        return f"gpt{m.group(1)}{_suf if _suf == 'sol' else _suf[:2]}"
    m = re.match(r"OLMo-([0-9]+)", n, re.I)
    if m:
        return f"olmo{m.group(1)}"
    m = re.match(r"Qwen[0-9.]*-([0-9.]+B)", n, re.I)
    if m:
        return f"q{m.group(1)}"
    m = re.match(r"gemma-([0-9]+)", n, re.I)
    if m:
        return f"gemma{m.group(1)}"
    return n[:12]

def failure_table(results, path=None, only=None, note=""):
    """Non-answers per examinee, by reason.

    Every one of these is scored as wrong, which is what the correlations in
    the other tables are computed on. That is the conservative reading: a
    refusal or a truncation is not evidence that the model could not do the
    item, so counting it as a failure can only weaken a correlation between a
    difficulty measure and accuracy, never manufacture one. It is recorded
    here so that a reader can see how much of a score is ability and how much
    is the model never having answered.

    The reasons, and what each means:

      cut          the reply hit the token budget. Measured, not inferred:
                   a model that finished emits its end token instead.
      refusal      the model declined. It is a property of the ITEM as much
                   as the model, and on this battery every refusal fell on
                   items built from reversed words, which look like an
                   obfuscation vector whatever the task is.
      timeout      no reply within the time limit.
      malformed    the reply was unusable, a tool call the model could not
                   make or output the parser could not read.
      other        anything else that came back empty.
    """
    def reason(r):
        f = str(r.get("finish") or "").upper()
        raw = str(r.get("raw") or "")
        if "REFUS" in f: return "refusal"
        if f in ("MAX_TOKENS", "LENGTH") or r.get("was_cut"): return "cut"
        if "Timeout" in raw or "408" in raw: return "timeout"
        if "MALFORMED" in raw.upper() or "MALFORMED" in f: return "malformed"
        return "other"

    if only is not None:
        results = [r for r in results if r["item"] in set(only)]
    by = collections.defaultdict(lambda: collections.Counter())
    tot = collections.Counter()
    for r in results:
        tot[r["examinee"]] += 1
        if not r.get("answer"):
            by[r["examinee"]][reason(r)] += 1

    names = by_capacity(set(tot), results)
    cols = ["cut", "refusal", "timeout", "malformed", "other"]
    print(f"{'examinee':22s} {'rows':>5s} " + "".join(f"{c:>10s}" for c in cols)
          + f"{'any':>7s}  score on the rest")
    rows = []
    for e in names:
        n = tot[e]
        miss = sum(by[e].values())
        ok = [r for r in results if r["examinee"] == e and r.get("answer")]
        s = (sum(r["n_correct"] for r in ok) / sum(r["k"] for r in ok)
             if ok else float("nan"))
        print(f"{short(e):22s} {n:5d} "
              + "".join(f"{by[e][c] or '':>10}" for c in cols)
              + f"{miss:7d}  {s:16.1%}")
        rows.append((e, n, dict(by[e]), miss, s))

    if path:
        esc = lambda t: str(t).replace("_", r"\_")
        L = [r"\begin{table}[t]", r"\centering",
             r"\caption{" + (note + " " if note else "")
             + r"Non-answers, by reason. Every one is scored as wrong, "
             r"which is what the correlations elsewhere are computed on. That "
             r"is conservative: a refusal or a truncation is not evidence that "
             r"the model could not do the item, so counting it as a failure "
             r"can only weaken a correlation between difficulty and accuracy, "
             r"never manufacture one. \textsf{cut} is measured from the finish "
             r"reason, since a model that finished emits its end token. "
             r"\textsf{refusal} is a property of the item as much as the "
             r"model: on this battery every refusal fell on items built from "
             r"reversed words, which look like an obfuscation vector whatever "
             r"the task is. The last column is the score on the rows that were "
             r"answered, which is what the examinee would have scored had "
             r"nothing gone wrong."
             + "}",
             r"\begin{tabular}{lr" + "r" * len(cols) + r"rr}", r"\toprule",
             "examinee & rows & " + " & ".join(r"\textsf{" + c + "}"
                                               for c in cols)
             + r" & any & score on the rest \\", r"\midrule"]
        for e, n, b, miss, s in rows:
            L.append(f"{esc(short(e))} & {n} & "
                     + " & ".join(str(b.get(c, "--") or "--") for c in cols)
                     + f" & {miss} & {s:.1%}".replace("%", r"\%") + r" \\")
        L += [r"\bottomrule", r"\end{tabular}", r"\end{table}"]
        open(path, "w").write("\n".join(L) + "\n")
        print(f"\nwrote {path}")
    return rows


In [ ]:
if ROWS_ALL_RAW:
    # SCOPED LIKE THE TABLE IT SITS BESIDE. ROWS_ALL_RAW is kept whole so that
    # an examinee dropped from the figures still has its non-answers counted,
    # but whole also meant every valid item at every prefix: a row count of 225
    # under a caption saying N = 42, and rows from prefixes this machine does
    # not use. The examinees stay; the items and the prefix are those of the
    # companion table.
    _unq = [i for i in ITEMS if ITEMS[i].get("n_M") is not None]
    _needR = {i: (ITEMS[i].get("n_M") or 0) for i in ITEMS}
    _bestR = {}
    for r in ROWS_ALL_RAW:
        _n = r.get("n_M")
        if _n is None or _n < _needR.get(r["item"], 0): continue
        _k = (r["examinee"], r["item"])
        _b = _bestR.get(_k)
        if _b is None or _n < _b: _bestR[_k] = _n
    ROWS_FAIL = [r for r in ROWS_ALL_RAW
                 if r.get("n_M") == _bestR.get((r["examinee"], r["item"]))]

    failure_table(ROWS_FAIL, only=_unq,
                  path=f"{FIGDIR}/tab_failures_{MACHINE_TAG}.tex",
                  note=f"$N = {len(_unq)}$: "
                       f"the valid items, unquestionable under {MACHINE_SHOWN}.")

    # and over the certified items alone, where h is a measurement rather
    # than the cost of writing the answer out
    _c = [i for i in ITEMS if ITEMS[i].get("band") not in (None, "p_lam")]
    if _c:
        print()
        failure_table(ROWS_FAIL, only=_c,
                      path=f"{FIGDIR}/tab_failures_comp_{MACHINE_TAG}.tex",
                      note=f"$N = {len(_c)}$: the valid items, unquestionable "
                           f"under {MACHINE_SHOWN} and certified under it.")
else:
    print("no results file")


## 6 · Scores

Who got what, and how much the repetitions agreed. Low agreement means the score is one coin toss rather than a measurement.

In [ ]:
# scores are in the cell above, as worst@3, mean@3 and best@3


## 7 · Everything written

The pngs go in a figure; the tex files are `\\input{}`-able so a number in the paper is never retyped.

In [ ]:
for f in sorted(glob.glob(f"{FIGDIR}/fig_*.png") + glob.glob(f"{FIGDIR}/tab_*.tex")):
    print(f"  {os.path.getsize(f):>9,d}  {f}")

# WHO BEHAVES ALIKE, and where the measures sit among them. An examinee is a
# vector over the items; two that find the same ones hard correlate whatever
# their level. The measures join the matrix with their sign flipped, since a
# dear item is a hard one, and the question the figure answers is whether a
# measure predicts the examinees about as well as they predict each other.
if ROWS:
    pairwise_matrix(ROWS, ITEMS, path=f"{FIGDIR}/fig_pairs{FIG_SUFFIX}.png",
                    title=f"Agreement between examinees, and the measures "
                          f"beside them\n{len(_unq)} items unquestionable "
                          f"under {MACHINE_SHOWN}")
    if len(_c) >= 6:
        pairwise_matrix([r for r in ROWS if r["item"] in _c], ITEMS,
                        path=f"{FIGDIR}/fig_pairs_comp{FIG_SUFFIX}.png",
                        title=f"Agreement between examinees, and the measures "
                              f"beside them\n{len(_c)} items certified "
                              f"under {MACHINE_SHOWN}")
